# 태양풍 속도 예측 — P12: P3 + P9(F3) + N1 재현 앙상블

`RUN_P11.md`의 사후분석을 반영한 제출 노트북입니다.

- P11의 실패 원인은 평균 자체가 아니라, P3 대체 멤버를 1–2 epoch만 학습해 P3 품질을 재현하지 못한 점입니다.
- P12는 기존 제출본 세 개를 **각각의 원래 학습 레시피**로 다시 학습합니다: P3의 validation early stopping, P9의 채택 F3 1 epoch, N1의 validation best checkpoint.
- 사전에 고정한 동일 가중치 평균만 사용합니다. validation 점수로 멤버나 가중치를 고르지 않습니다.

예상 public RMSE는 기존 제출본 오차 상관으로 역산한 56.72 (P3 대비 −2.08)입니다. 이는 추정치이며, 새 학습 실행의 난수/하드웨어 차이에 따라 달라질 수 있습니다.


## 1. P3 — 원래 early-stopping 레시피 재현

In [ ]:
P3 = {}
exec('# ===== copied from code_p3_fix.ipynb, cell 2 =====\n\nfrom pathlib import Path\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nSEED = 777\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\nDATA_ROOT_CANDIDATES = [\n    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,\n    Path("public_dataset/competition_dataset_6h"),\n    Path("/home/jovyan/public_dataset/competition_dataset_6h"),\n    Path("public/public_dataset/competition_dataset_6h"),\n    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),\n    Path("dataset"),\n    Path("/home/jovyan/dataset"),\n]\nDATA_ROOT = None\nfor candidate in DATA_ROOT_CANDIDATES:\n    if candidate is not None and (candidate / "train/inputs.csv").exists():\n        DATA_ROOT = candidate\n        break\nif DATA_ROOT is None:\n    searched = "\\n".join(f"  - {c}" for c in DATA_ROOT_CANDIDATES if c is not None)\n    raise FileNotFoundError("데이터 경로를 찾지 못했습니다:\\n" + searched)\n\nWORK_DIR = Path("work")\nCACHE_ROOT = WORK_DIR / "cache"\nOUTPUT_DIR = WORK_DIR / "outputs_p3"\nSUBMISSION_DIR = Path("submission")\nfor directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):\n    directory.mkdir(parents=True, exist_ok=True)\n\nIMAGE_SIZE = 128\nCHANNELS = ("193", "211")\nBATCH_SIZE = 64\nEPOCHS = 60\nLEARNING_RATE = 3e-4\nWEIGHT_DECAY = 1e-3\nGRAD_CLIP = 1.0\nSCHEDULER_PATIENCE = 3\nEARLY_STOP_PATIENCE = 10\nNUM_WORKERS = 4\nLOSS_EPSILON = 1e-8\nLOSS_SCALE = 100.0\nLOSS_MODE = "metric"\n\n# --- 브랜치 스위치 (ablation 용) ---------------------------------------\nUSE_CNN = False    # 3D CNN 영상 브랜치. P3 기본값은 끔 (과적합 주범)\nUSE_CH = True      # 코로나홀 격자 피처 브랜치\nUSE_BALLISTIC = True   # horizon별 탄도 정렬 피처\n\n# --- 코로나홀 추출 (Collin 2025) ---------------------------------------\nCH_GRID = (3, 5)          # (위도 구간, 경도 구간). 논문은 4x3 이 timeline RMSE 최적\nCH_THRESHOLD_RATIO = 0.45  # 원반 중앙값 대비 이 비율보다 어두우면 코로나홀\nDISK_MARGIN = 0.95         # 림 밝아짐(limb brightening) 회피용 반지름 축소\nTRANSIT_SPEEDS = (350.0, 500.0, 700.0)   # 탄도 역산에 쓸 가정 속도 (km/s)\nAU_KM = 1.496e8\n\nDROPOUT = 0.4\nHORIZON_EMBED = 8\nAUGMENT = True\nAUG_BRIGHTNESS = 0.10\nAUG_SHIFT_PIXELS = 6\nAUG_NOISE_STD = 0.02\nAUG_ERASE_PROB = 0.3\nAUG_CH_NOISE = 0.0        # CH 피처에 주는 곱셈 노이즈\nCH_DROPOUT = 0.15          # CH 브랜치 전용 dropout\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nUSE_AMP = DEVICE.type == "cuda"\nPIN_MEMORY = DEVICE.type == "cuda"\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\nif DEVICE.type == "cuda":\n    torch.backends.cudnn.benchmark = True\nelse:\n    print("WARNING: CUDA Unavailable")\n\nprint("PyTorch:", torch.__version__, "| device:", DEVICE)\nif DEVICE.type == "cuda":\n    print("GPU:", torch.cuda.get_device_name(0))\nprint("data:", DATA_ROOT.resolve())\nprint(f"branches: CNN={USE_CNN} CH={USE_CH} BALLISTIC={USE_BALLISTIC}")\n\n\n# ===== copied from code_p3_fix.ipynb, cell 4 =====\n\nIMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]\nWIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]\nTARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]\nHORIZONS = np.arange(1, 13) * 6\n\ntrain_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")\ntrain_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")\nval_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")\nval_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")\ntest_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")\ntest_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")\n\nassert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()\nassert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()\nassert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()\nassert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)\nassert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)\nassert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)\nassert not any(column.startswith("target_") for column in test_inputs.columns)\n\n\ndef forward_fill_rows(values):\n    valid = np.isfinite(values)\n    positions = np.where(valid, np.arange(values.shape[1])[None, :], 0)\n    np.maximum.accumulate(positions, axis=1, out=positions)\n    rows = np.arange(values.shape[0])[:, None]\n    return np.where(valid.any(axis=1, keepdims=True), values[rows, positions], values)\n\n\ndef fill_wind(frame, fallback):\n    values = frame[WIND_COLUMNS].to_numpy(np.float32)\n    valid = np.isfinite(values).astype(np.float32)\n    filled = forward_fill_rows(values)\n    filled = forward_fill_rows(filled[:, ::-1])[:, ::-1]\n    filled = np.where(np.isfinite(filled), filled, fallback)\n    return np.ascontiguousarray(filled), np.ascontiguousarray(valid)\n\n\nWIND_FALLBACK = float(np.nanmedian(train_inputs[WIND_COLUMNS].to_numpy(np.float32)))\ntrain_wind, train_wind_valid = fill_wind(train_inputs, WIND_FALLBACK)\nval_wind, val_wind_valid = fill_wind(val_inputs, WIND_FALLBACK)\ntest_wind, test_wind_valid = fill_wind(test_inputs, WIND_FALLBACK)\n\ntrain_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\nval_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\nassert np.isfinite(train_targets).all() and np.isfinite(val_targets).all()\n\nprint("samples:", len(train_inputs), len(val_inputs), len(test_inputs))\nprint(f"train wind mean={train_wind.mean():.1f} target mean={train_targets.mean():.1f}")\n\n\n# ===== copied from code_p3_fix.ipynb, cell 6 =====\n\ndef prepare_image_memmap(split, inputs):\n    image_root = DATA_ROOT / split\n    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"\n    cache_root.mkdir(parents=True, exist_ok=True)\n    array_path = cache_root / f"{split}_images.npy"\n    metadata_path = cache_root / f"{split}_metadata.json"\n    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())\n    expected = {"image_size": IMAGE_SIZE, "channels": list(CHANNELS), "filenames": filenames}\n    shape = (len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE)\n\n    valid = False\n    if array_path.exists() and metadata_path.exists():\n        try:\n            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n            cached = np.load(array_path, mmap_mode="r")\n            valid = (metadata == expected and cached.shape == shape\n                     and cached.dtype == np.uint8)\n        except (OSError, ValueError, json.JSONDecodeError):\n            valid = False\n\n    if not valid:\n        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")\n        metadata_temp = metadata_path.with_name(metadata_path.name + f".partial.{os.getpid()}")\n        resized = np.lib.format.open_memmap(array_temp, mode="w+", dtype=np.uint8, shape=shape)\n        resampling = Image.Resampling.BILINEAR\n        for index, filename in enumerate(filenames):\n            for channel_index, channel in enumerate(CHANNELS):\n                with Image.open(image_root / channel / filename) as image:\n                    resized[index, channel_index] = np.asarray(\n                        image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), resampling),\n                        dtype=np.uint8)\n            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):\n                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)\n        resized.flush()\n        del resized\n        metadata_temp.write_text(json.dumps(expected, ensure_ascii=False) + "\\n", encoding="utf-8")\n        array_temp.replace(array_path)\n        metadata_temp.replace(metadata_path)\n        print(f"created cache: {array_path.resolve()}")\n    else:\n        print(f"reusing cache: {array_path.resolve()}")\n\n    image_array = np.load(array_path, mmap_mode="r")\n    return image_array, {name: i for i, name in enumerate(filenames)}\n\n\ntrain_image_array, train_image_index = prepare_image_memmap("train", train_inputs)\nval_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)\ntest_image_array, test_image_index = prepare_image_memmap("test", test_inputs)\nprint("고유 이미지:", len(train_image_index), len(val_image_index), len(test_image_index))\n\n\n# ===== copied from code_p3_fix.ipynb, cell 8 =====\n\ndef detect_disk(image_array, sample_count=400):\n    indexes = np.unique(np.linspace(0, len(image_array) - 1, sample_count).astype(int))\n    mean_image = np.asarray(image_array[indexes], dtype=np.float64).mean(axis=(0, 1))\n    # 배경은 어둡고 원반은 밝습니다. 단순 임계로 원반 픽셀을 잡습니다.\n    mask = mean_image > mean_image.max() * 0.15\n    ys, xs = np.nonzero(mask)\n    center_y, center_x = float(ys.mean()), float(xs.mean())\n    radius = float(np.sqrt(mask.sum() / np.pi))\n    return center_y, center_x, radius, mean_image\n\n\nDISK_Y, DISK_X, DISK_R, MEAN_IMAGE = detect_disk(train_image_array)\nEFFECTIVE_R = DISK_R * DISK_MARGIN\nprint(f"원반 검출: center=({DISK_Y:.1f}, {DISK_X:.1f}) radius={DISK_R:.1f}px "\n      f"(유효 {EFFECTIVE_R:.1f}px, 프레임의 {2*DISK_R/IMAGE_SIZE:.0%})")\n\ngrid_y, grid_x = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE].astype(np.float64)\nradius_map = np.sqrt((grid_y - DISK_Y) ** 2 + (grid_x - DISK_X) ** 2)\nDISK_MASK = radius_map <= EFFECTIVE_R\n\n# 원반 외접 사각형을 (위도, 경도) 격자로 분할\nn_lat, n_lon = CH_GRID\nlat_edge = np.clip(((grid_y - (DISK_Y - EFFECTIVE_R)) / (2 * EFFECTIVE_R) * n_lat), 0, n_lat - 1e-6)\nlon_edge = np.clip(((grid_x - (DISK_X - EFFECTIVE_R)) / (2 * EFFECTIVE_R) * n_lon), 0, n_lon - 1e-6)\nCELL_ID = (lat_edge.astype(np.int64) * n_lon + lon_edge.astype(np.int64))\nCELL_ID_FLAT = CELL_ID[DISK_MASK]\nN_CELLS = n_lat * n_lon\nCELL_COUNTS = np.bincount(CELL_ID_FLAT, minlength=N_CELLS).astype(np.float32)\nCELL_COUNTS = np.maximum(CELL_COUNTS, 1.0)\n# 경도 중앙 열(자오선)과 적도 위도대의 셀 인덱스\nCENTRAL_LON = n_lon // 2\nEQUATOR_LAT = n_lat // 2\nCENTRAL_CELL = EQUATOR_LAT * n_lon + CENTRAL_LON\nprint(f"격자 {n_lat}x{n_lon} = {N_CELLS} 셀, 원반 픽셀 {int(DISK_MASK.sum()):,}개, "\n      f"중앙자오선 셀 index={CENTRAL_CELL}")\n\n\ndef compute_ch_grid(image_array, chunk=256):\n    # 반환: (n_images, N_CELLS) 셀별 코로나홀 면적 비율\n    result = np.zeros((len(image_array), N_CELLS), dtype=np.float32)\n    onehot = np.zeros((len(CELL_ID_FLAT), N_CELLS), dtype=np.float32)\n    onehot[np.arange(len(CELL_ID_FLAT)), CELL_ID_FLAT] = 1.0\n    for start in range(0, len(image_array), chunk):\n        block = np.asarray(image_array[start:start + chunk], dtype=np.float32)\n        on_disk = block[:, :, DISK_MASK]                       # (n, 2, npix)\n        median = np.median(on_disk, axis=2, keepdims=True)     # (n, 2, 1)\n        dark = on_disk < (CH_THRESHOLD_RATIO * median)\n        # 193 과 211 두 채널 모두에서 어두운 픽셀만 코로나홀로 인정\n        coronal_hole = np.logical_and(dark[:, 0], dark[:, 1]).astype(np.float32)\n        result[start:start + chunk] = (coronal_hole @ onehot) / CELL_COUNTS\n    return result\n\n\ndef cached_ch_grid(split, image_array):\n    path = CACHE_ROOT / f"ch_{split}_{n_lat}x{n_lon}_{CH_THRESHOLD_RATIO}_{IMAGE_SIZE}.npy"\n    if path.exists():\n        grid = np.load(path)\n        if grid.shape == (len(image_array), N_CELLS):\n            print(f"reusing CH cache: {path.name}")\n            return grid\n    grid = compute_ch_grid(image_array)\n    np.save(path, grid)\n    print(f"created CH cache: {path.name}  shape={grid.shape}")\n    return grid\n\n\ntrain_ch = cached_ch_grid("train", train_image_array)\nval_ch = cached_ch_grid("validation", val_image_array)\ntest_ch = cached_ch_grid("test", test_image_array)\n\nprint(f"\\nCH 면적 비율 — train 전체 평균 {train_ch.mean():.4f}, "\n      f"중앙자오선 셀 평균 {train_ch[:, CENTRAL_CELL].mean():.4f}")\n\nfigure, axes = plt.subplots(1, 3, figsize=(13, 4))\naxes[0].imshow(MEAN_IMAGE, cmap="gray")\ncircle = plt.Circle((DISK_X, DISK_Y), EFFECTIVE_R, fill=False, color="red", linewidth=1.5)\naxes[0].add_patch(circle)\naxes[0].set_title("train 평균 영상 + 검출된 원반")\nsample_image = np.asarray(train_image_array[0], dtype=np.float32)\nsample_dark = sample_image < (CH_THRESHOLD_RATIO * np.median(sample_image[:, DISK_MASK], axis=1)[:, None, None])\naxes[1].imshow(np.logical_and(sample_dark[0], sample_dark[1]) & DISK_MASK, cmap="gray")\naxes[1].set_title("코로나홀 마스크 예시 (193 AND 211)")\naxes[2].imshow(train_ch[:400].T, aspect="auto", cmap="viridis")\naxes[2].set_title("셀별 CH 면적 (앞 400시점)")\naxes[2].set_xlabel("time index"); axes[2].set_ylabel("cell")\nfor axis in axes[:2]:\n    axis.set_xticks([]); axis.set_yticks([])\nplt.tight_layout(); plt.show()\n\n\n# ===== copied from code_p3_fix.ipynb, cell 10 =====\n\ndef compute_image_stats(array, chunk=256):\n    total = np.zeros(len(CHANNELS), np.float64)\n    total_square = np.zeros(len(CHANNELS), np.float64)\n    count = 0\n    for start in range(0, len(array), chunk):\n        block = np.asarray(array[start:start + chunk], dtype=np.float64) / 255.0\n        total += block.sum(axis=(0, 2, 3))\n        total_square += (block ** 2).sum(axis=(0, 2, 3))\n        count += block.shape[0] * block.shape[2] * block.shape[3]\n    mean = total / count\n    return mean.astype(np.float32), np.sqrt(\n        np.maximum(total_square / count - mean ** 2, 1e-12)).astype(np.float32)\n\n\nIMAGE_MEAN, IMAGE_STD = compute_image_stats(train_image_array)\nWIND_MEAN = float(train_wind.mean())\nWIND_STD = float(train_wind.std() + 1e-6)\nDIFF_STD = float(np.diff(train_wind, axis=1, prepend=train_wind[:, :1]).std() + 1e-6)\nCH_MEAN = train_ch.mean(axis=0).astype(np.float32)\nCH_STD = (train_ch.std(axis=0) + 1e-8).astype(np.float32)\n\ntrain_residual = train_targets - train_wind[:, -1:]\nRESIDUAL_MEAN = train_residual.mean(axis=0).astype(np.float32)\nRESIDUAL_STD = (train_residual.std(axis=0) + 1e-6).astype(np.float32)\nCLIP_LOW = float(train_targets.min() * 0.95)\nCLIP_HIGH = float(train_targets.max() * 1.05)\n\n\ndef ballistic_indices():\n    # (12, n_speeds) 윈도우 내 실수 인덱스. 인덱스 19 가 마지막 관측 시점 T0.\n    table = np.zeros((12, len(TRANSIT_SPEEDS)), dtype=np.float32)\n    for horizon_index in range(12):\n        lead_hours = (horizon_index + 1) * 6.0\n        for speed_index, speed in enumerate(TRANSIT_SPEEDS):\n            transit_hours = AU_KM / speed / 3600.0\n            table[horizon_index, speed_index] = np.clip(\n                19.0 + (lead_hours - transit_hours) / 6.0, 0.0, 19.0)\n    return table\n\n\nBALLISTIC_INDEX = ballistic_indices()\nN_SPEEDS = len(TRANSIT_SPEEDS)\n\n\ndef image_index_matrix(inputs, image_index):\n    return np.asarray([\n        [image_index[name] for name in row]\n        for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)\n    ], dtype=np.int32)\n\n\ndef compute_ballistic(ch_grid, indexes):\n    # 중앙자오선 셀 시계열을 탄도 역산 인덱스에서 선형보간 -> (n, 12, n_speeds)\n    central = ch_grid[indexes][:, :, CENTRAL_CELL]\n    lower = np.floor(BALLISTIC_INDEX).astype(np.int64)\n    upper = np.minimum(lower + 1, 19)\n    weight = (BALLISTIC_INDEX - lower).astype(np.float32)\n    return (central[:, lower] * (1.0 - weight) + central[:, upper] * weight).astype(np.float32)\n\n\n# 탄도 피처 정규화 통계도 반드시 train split 에서만 산출합니다.\n_train_ballistic = compute_ballistic(train_ch, image_index_matrix(train_inputs, train_image_index))\nBALLISTIC_MEAN = float(_train_ballistic.mean())\nBALLISTIC_STD = float(_train_ballistic.std() + 1e-8)\n\nprint("탄도 역산 — 가정 속도별 전달 시간:")\nfor speed in TRANSIT_SPEEDS:\n    print(f"  v={speed:5.0f} km/s -> tau = {AU_KM / speed / 3600.0:5.1f} h "\n          f"({AU_KM / speed / 86400.0:.2f} 일)")\nprint("\\nhorizon별 근원 시점 인덱스 (19 = 마지막 관측):")\nprint(pd.DataFrame(BALLISTIC_INDEX, index=[f"{h}h" for h in HORIZONS],\n                   columns=[f"v={int(v)}" for v in TRANSIT_SPEEDS]).round(2))\nprint(f"\\nresidual std by horizon: {np.round(RESIDUAL_STD, 1)}")\nprint(f"clip range: [{CLIP_LOW:.1f}, {CLIP_HIGH:.1f}] km/s")\n\n\n# ===== copied from code_p3_fix.ipynb, cell 12 =====\n\nSTAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]\n_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5\n_TIME_DENOMINATOR = float((_TIME_CENTERED ** 2).sum())\n\n\ndef build_wind_stats(wind):\n    last = wind[:, -1]\n    mean4 = wind[:, -4:].mean(axis=1)\n    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOMINATOR\n    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),\n                     wind.max(axis=1), slope, last - mean4,\n                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)\n\n\ntrain_stats_raw = build_wind_stats(train_wind)\nSTATS_MEAN = train_stats_raw.mean(axis=0).astype(np.float32)\nSTATS_STD = (train_stats_raw.std(axis=0) + 1e-6).astype(np.float32)\nNUM_STATS = len(STAT_NAMES)\n\n\nclass SolarWindDataset(Dataset):\n    def __init__(self, image_array, image_index, inputs, wind, wind_valid,\n                 ch_grid, targets=None, training=False):\n        self.training = training\n        self.image_array = image_array\n        self.image_indexes = image_index_matrix(inputs, image_index)\n        self.sample_ids = inputs.sample_id.to_numpy()\n        self.last_wind = np.ascontiguousarray(wind[:, -1]).astype(np.float32)\n        self.wind_seq = np.stack([\n            (wind - WIND_MEAN) / WIND_STD,\n            np.diff(wind, axis=1, prepend=wind[:, :1]) / DIFF_STD,\n            wind_valid,\n        ], axis=2).astype(np.float32)\n        self.wind_stats = ((build_wind_stats(wind) - STATS_MEAN) / STATS_STD).astype(np.float32)\n\n        # (n_samples, 20, N_CELLS) 시퀀스로 미리 펼쳐 둡니다.\n        self.ch_seq = ((ch_grid[self.image_indexes] - CH_MEAN) / CH_STD).astype(np.float32)\n        # 탄도 정렬 피처. 정규화는 train 통계(BALLISTIC_MEAN/STD)로 고정합니다.\n        self.ballistic = ((compute_ballistic(ch_grid, self.image_indexes) - BALLISTIC_MEAN)\n                          / BALLISTIC_STD).astype(np.float32)           # (n, 12, n_speeds)\n\n        self.targets = targets.astype(np.float32) if targets is not None else None\n        self.image_mean = IMAGE_MEAN.reshape(1, len(CHANNELS), 1, 1)\n        self.image_std = IMAGE_STD.reshape(1, len(CHANNELS), 1, 1)\n\n    def __len__(self):\n        return len(self.sample_ids)\n\n    def __getitem__(self, item):\n        if USE_CNN:\n            images = np.asarray(\n                self.image_array[self.image_indexes[item]], dtype=np.float32) / 255.0\n            images = ((images - self.image_mean) / self.image_std).astype(np.float32)\n        else:\n            images = np.zeros((1, 1, 1, 1), dtype=np.float32)\n\n        ch_seq = self.ch_seq[item]\n        ballistic = self.ballistic[item]\n        if self.training and AUGMENT and AUG_CH_NOISE > 0:\n            # CH 피처에도 약한 곱셈 노이즈를 줘 브랜치 과적합을 억제합니다.\n            ch_seq = ch_seq * (1.0 + np.random.normal(0, AUG_CH_NOISE, ch_seq.shape)\n                               ).astype(np.float32)\n            ballistic = ballistic * (1.0 + np.random.normal(\n                0, AUG_CH_NOISE, ballistic.shape)).astype(np.float32)\n\n        result = {\n            "images": torch.from_numpy(np.ascontiguousarray(images)),\n            "wind_seq": torch.from_numpy(self.wind_seq[item]),\n            "wind_stats": torch.from_numpy(self.wind_stats[item]),\n            "ch_seq": torch.from_numpy(np.ascontiguousarray(ch_seq)),\n            "ballistic": torch.from_numpy(np.ascontiguousarray(ballistic)),\n            "last_wind": torch.tensor(self.last_wind[item]),\n            "sample_id": self.sample_ids[item],\n        }\n        if self.targets is not None:\n            result["target"] = torch.from_numpy(self.targets[item])\n        return result\n\n\ndef seed_worker(worker_id):\n    worker_seed = (SEED + worker_id) % (2 ** 32)\n    random.seed(worker_seed)\n    np.random.seed(worker_seed)\n\n\ndef make_loader(dataset, shuffle):\n    options = dict(dataset=dataset, batch_size=BATCH_SIZE, shuffle=shuffle,\n                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False,\n                   worker_init_fn=seed_worker,\n                   generator=torch.Generator().manual_seed(SEED))\n    if NUM_WORKERS > 0:\n        options.update(persistent_workers=True, prefetch_factor=2)\n    return DataLoader(**options)\n\n\ntrain_dataset = SolarWindDataset(train_image_array, train_image_index, train_inputs,\n                                 train_wind, train_wind_valid, train_ch,\n                                 train_targets, training=True)\nval_dataset = SolarWindDataset(val_image_array, val_image_index, val_inputs,\n                               val_wind, val_wind_valid, val_ch, val_targets)\ntrain_loader = make_loader(train_dataset, shuffle=True)\nval_loader = make_loader(val_dataset, shuffle=False)\n\nbatch = next(iter(train_loader))\nassert batch["ch_seq"].shape[1:] == (20, N_CELLS)\nassert batch["ballistic"].shape[1:] == (12, N_SPEEDS)\nprint({k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})\n\n\n# ===== copied from code_p3_fix.ipynb, cell 14 =====\n\nclass Inception3D(nn.Module):\n    def __init__(self, in_channels, branch_channels=32):\n        super().__init__()\n        self.branch_1 = nn.Sequential(\n            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True))\n        self.branch_3 = nn.Sequential(\n            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),\n            nn.Conv3d(branch_channels, branch_channels, (1, 3, 3), padding=(0, 1, 1)),\n            nn.ReLU(inplace=True))\n        self.branch_5 = nn.Sequential(\n            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),\n            nn.Conv3d(branch_channels, branch_channels, (1, 5, 5), padding=(0, 2, 2)),\n            nn.ReLU(inplace=True))\n        self.branch_pool = nn.Sequential(\n            nn.MaxPool3d((1, 3, 3), stride=1, padding=(0, 1, 1)),\n            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True))\n\n    def forward(self, x):\n        return torch.cat([self.branch_1(x), self.branch_3(x),\n                          self.branch_5(x), self.branch_pool(x)], dim=1)\n\n\nclass SolarWindP3(nn.Module):\n    def __init__(self):\n        super().__init__()\n        shared_dim = 0\n\n        self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)\n        self.stats_encoder = nn.Sequential(\n            nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),\n            nn.Linear(128, 64), nn.SELU(inplace=True))\n        shared_dim += 96 + 64\n\n        if USE_CH:\n            self.ch_gru = nn.GRU(N_CELLS, 64, num_layers=2, batch_first=True)\n            self.ch_dropout = nn.Dropout(CH_DROPOUT)\n            shared_dim += 64\n\n        if USE_CNN:\n            self.stem = nn.Sequential(\n                nn.Conv3d(len(CHANNELS), 32, (1, 5, 5), padding=(0, 2, 2)),\n                nn.BatchNorm3d(32), nn.ReLU(inplace=True),\n                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),\n                nn.Conv3d(32, 64, (1, 3, 3), padding=(0, 1, 1)),\n                nn.BatchNorm3d(64), nn.ReLU(inplace=True),\n                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)))\n            blocks, in_channels = [], 64\n            for _ in range(3):\n                blocks.extend([Inception3D(in_channels, 32),\n                               nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))])\n                in_channels = 128\n            self.image_encoder = nn.Sequential(*blocks)\n            self.image_lstm = nn.LSTM(128 * 1 * 4, 128, batch_first=True)\n            self.image_dropout = nn.Dropout(DROPOUT)\n            shared_dim += 128\n\n        self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)\n        head_input = shared_dim + HORIZON_EMBED + (N_SPEEDS if USE_BALLISTIC else 0)\n        # head 는 12 horizon 에 동일 가중치로 적용됩니다 (nn.Linear 는 마지막 축에만 작용).\n        self.head = nn.Sequential(\n            nn.Linear(head_input, 192), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),\n            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),\n            nn.Linear(96, 1))\n\n        self.register_buffer("residual_mean", torch.as_tensor(RESIDUAL_MEAN))\n        self.register_buffer("residual_std", torch.as_tensor(RESIDUAL_STD))\n        print(f"shared_dim={shared_dim}  head_input={head_input}")\n\n    def forward(self, images, wind_seq, wind_stats, ch_seq, ballistic):\n        _, wind_hidden = self.wind_gru(wind_seq)\n        parts = [F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats)]\n\n        if USE_CH:\n            _, ch_hidden = self.ch_gru(ch_seq)\n            parts.append(self.ch_dropout(F.relu(ch_hidden[-1])))\n\n        if USE_CNN:\n            features = images.permute(0, 2, 1, 3, 4).contiguous()\n            features = self.image_encoder(self.stem(features))\n            features = F.adaptive_avg_pool3d(features, (features.shape[2], 1, 4))\n            features = features.permute(0, 2, 1, 3, 4).flatten(2)\n            _, (hidden, _) = self.image_lstm(features)\n            parts.append(self.image_dropout(F.relu(hidden[-1])))\n\n        shared = torch.cat(parts, dim=1)                                   # (B, D)\n        batch_size = shared.shape[0]\n        expanded = shared.unsqueeze(1).expand(batch_size, 12, shared.shape[1])\n        embedding = self.horizon_embedding.unsqueeze(0).expand(batch_size, 12, HORIZON_EMBED)\n        head_parts = [expanded, embedding]\n        if USE_BALLISTIC:\n            head_parts.append(ballistic)\n        z = self.head(torch.cat(head_parts, dim=2)).squeeze(-1)            # (B, 12)\n        return z * self.residual_std + self.residual_mean\n\n\ndef build_model():\n    return SolarWindP3().to(DEVICE)\n\n\nmodel = build_model()\nprint("trainable parameters:",\n      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")\n\n\n# ===== copied from code_p3_fix.ipynb, cell 16 =====\n\ndef official_rmse(y_true, y_pred):\n    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))\n    return float(per_horizon.mean()), per_horizon\n\n\ndef pooled_rmse(y_true, y_pred):\n    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))\n\n\ndef metrics_by_horizon(y_true, y_pred, persistence=None):\n    rows = []\n    for index in range(12):\n        actual, predicted = y_true[:, index], y_pred[:, index]\n        error = predicted - actual\n        denominator = np.std(actual) * np.std(predicted)\n        row = {"horizon_h": int(HORIZONS[index]),\n               "rmse": float(np.sqrt(np.mean(error ** 2))),\n               "mae": float(np.mean(np.abs(error))),\n               "corr": float(np.corrcoef(actual, predicted)[0, 1]) if denominator > 0 else np.nan}\n        if persistence is not None:\n            row["persistence_rmse"] = float(np.sqrt(np.mean((persistence[:, index] - actual) ** 2)))\n            row["gain"] = row["persistence_rmse"] - row["rmse"]\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef metric_loss(prediction, target):\n    error = (prediction - target) / LOSS_SCALE\n    if LOSS_MODE == "mse":\n        return (error ** 2).mean()\n    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()\n\n\ndef augment_batch(images):\n    # GPU 에서 수행합니다. numpy 증강은 CPU 병목으로 epoch 시간이 4배 늘었습니다.\n    if AUG_SHIFT_PIXELS > 0:\n        shift_y = int(torch.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1, (1,)).item())\n        shift_x = int(torch.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1, (1,)).item())\n        images = torch.roll(images, shifts=(shift_y, shift_x), dims=(3, 4))\n    count = images.shape[0]\n    scale = 1.0 + (torch.rand(count, 1, 1, 1, 1, device=images.device) * 2 - 1) * AUG_BRIGHTNESS\n    offset = torch.randn(count, 1, 1, 1, 1, device=images.device) * AUG_BRIGHTNESS\n    images = images * scale + offset\n    if AUG_NOISE_STD > 0:\n        images = images + torch.randn_like(images) * AUG_NOISE_STD\n    if AUG_ERASE_PROB > 0:\n        size = max(4, IMAGE_SIZE // 8)\n        selected = torch.rand(count, device=images.device) < AUG_ERASE_PROB\n        if bool(selected.any()):\n            top = int(torch.randint(0, IMAGE_SIZE - size, (1,)).item())\n            left = int(torch.randint(0, IMAGE_SIZE - size, (1,)).item())\n            images[selected, :, :, top:top + size, left:left + size] = 0.0\n    return images\n\n\nval_persistence = np.repeat(val_wind[:, -1:], 12, axis=1).astype(np.float64)\npersistence_score, persistence_per_horizon = official_rmse(val_targets, val_persistence)\nprint(f"[기준선] persistence 공식 RMSE = {persistence_score:.3f} km/s")\nprint("horizon별:", np.round(persistence_per_horizon, 1))\n\n\n# ===== copied from code_p3_fix.ipynb, cell 18 =====\n\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\nmodel = build_model()\noptimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)\nscheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n    optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE, min_lr=1e-6)\nscaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)\n\ncheckpoint_path = OUTPUT_DIR / "best_model.pth"\nif checkpoint_path.exists():\n    checkpoint_path.unlink()\nbest_val_score = float("inf")\nepochs_without_improvement = 0\nhistory = []\n\nCONFIG = {"image_size": IMAGE_SIZE, "channels": list(CHANNELS), "use_cnn": USE_CNN,\n          "use_ch": USE_CH, "use_ballistic": USE_BALLISTIC, "ch_grid": list(CH_GRID),\n          "ch_threshold_ratio": CH_THRESHOLD_RATIO, "disk": [DISK_Y, DISK_X, DISK_R],\n          "transit_speeds": list(TRANSIT_SPEEDS), "seed": SEED,\n          "image_mean": IMAGE_MEAN.tolist(), "image_std": IMAGE_STD.tolist(),\n          "wind_mean": WIND_MEAN, "wind_std": WIND_STD, "diff_std": DIFF_STD,\n          "ch_mean": CH_MEAN.tolist(), "ch_std": CH_STD.tolist(),\n          "stats_mean": STATS_MEAN.tolist(), "stats_std": STATS_STD.tolist(),\n          "residual_mean": RESIDUAL_MEAN.tolist(), "residual_std": RESIDUAL_STD.tolist(),\n          "clip_low": CLIP_LOW, "clip_high": CLIP_HIGH,\n          "initialization": "random_from_scratch"}\n\n\ndef run_epoch(loader, training):\n    model.train(training)\n    squared_error_sum = np.zeros(12, dtype=np.float64)\n    sample_count = 0\n    for batch in loader:\n        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)\n        ch_seq = batch["ch_seq"].to(DEVICE, non_blocking=PIN_MEMORY)\n        ballistic = batch["ballistic"].to(DEVICE, non_blocking=PIN_MEMORY)\n        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)\n        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)\n\n        if training and USE_CNN and AUGMENT:\n            images = augment_batch(images)\n        if training:\n            optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training):\n            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n                residual = model(images, wind_seq, wind_stats, ch_seq, ballistic)\n            prediction = residual.float() + last_wind.unsqueeze(1)\n            loss = metric_loss(prediction, target)\n            if training:\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n                scaler.step(optimizer)\n                scaler.update()\n\n        error = (prediction.detach() - target).double()\n        squared_error_sum += torch.sum(error ** 2, dim=0).cpu().numpy()\n        sample_count += error.shape[0]\n    per_horizon = np.sqrt(squared_error_sum / sample_count)\n    return float(per_horizon.mean()), per_horizon\n\n\nfor epoch in range(1, EPOCHS + 1):\n    started = time.perf_counter()\n    train_score, _ = run_epoch(train_loader, training=True)\n    with torch.no_grad():\n        val_score, _ = run_epoch(val_loader, training=False)\n    scheduler.step(val_score)\n    learning_rate = optimizer.param_groups[0]["lr"]\n    elapsed = time.perf_counter() - started\n    history.append({"epoch": epoch, "train_rmse": train_score, "val_rmse": val_score,\n                    "learning_rate": learning_rate, "seconds": elapsed})\n    marker = ""\n    if val_score < best_val_score:\n        best_val_score = val_score\n        epochs_without_improvement = 0\n        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,\n                    "val_official_rmse": val_score, **CONFIG}, checkpoint_path)\n        marker = "  <- best"\n    else:\n        epochs_without_improvement += 1\n    print(f"epoch={epoch:03d} train={train_score:7.3f} val={val_score:7.3f} "\n          f"lr={learning_rate:.2e} {elapsed:6.1f}s{marker}", flush=True)\n    if epochs_without_improvement >= EARLY_STOP_PATIENCE:\n        print("early stopping")\n        break\n\nhistory_frame = pd.DataFrame(history)\nhistory_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)\nfigure, axis = plt.subplots(figsize=(7, 4))\naxis.plot(history_frame.epoch, history_frame.train_rmse, label="train")\naxis.plot(history_frame.epoch, history_frame.val_rmse, label="validation")\naxis.axhline(persistence_score, color="gray", linestyle="--", label="persistence")\naxis.set_xlabel("epoch"); axis.set_ylabel("official RMSE (km/s)")\naxis.grid(alpha=0.3); axis.legend()\nplt.tight_layout(); plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=140); plt.show()\n\ncheckpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)\nmodel.load_state_dict(checkpoint["model_state_dict"])\nprint(f"best epoch {checkpoint[\'epoch\']}  val official RMSE {checkpoint[\'val_official_rmse\']:.3f}")\n\n\n# ===== copied from code_p3_fix.ipynb, cell 20 =====\n\n@torch.no_grad()\ndef predict(loader):\n    model.eval()\n    predictions, sample_ids = [], []\n    for batch in loader:\n        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)\n        ch_seq = batch["ch_seq"].to(DEVICE, non_blocking=PIN_MEMORY)\n        ballistic = batch["ballistic"].to(DEVICE, non_blocking=PIN_MEMORY)\n        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)\n        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n            residual = model(images, wind_seq, wind_stats, ch_seq, ballistic)\n        prediction = (residual.float() + last_wind.unsqueeze(1)).clamp(CLIP_LOW, CLIP_HIGH)\n        predictions.append(prediction.cpu().numpy())\n        sample_ids.extend(batch["sample_id"])\n    return np.concatenate(predictions).astype(np.float64), sample_ids\n\n\nvalidation_prediction, validation_ids = predict(val_loader)\nassert validation_ids == val_inputs.sample_id.tolist()\nmodel_score, _ = official_rmse(val_targets, validation_prediction)\nvalidation_metrics = metrics_by_horizon(val_targets, validation_prediction, val_persistence)\nvalidation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)\n\nprint(f"공식 RMSE (mean of horizon RMSE) : {model_score:8.3f} km/s")\nprint(f"pooled RMSE (전체 원소)          : {pooled_rmse(val_targets, validation_prediction):8.3f} km/s")\nprint(f"persistence 공식 RMSE            : {persistence_score:8.3f} km/s")\nprint(f"persistence 대비 개선             : {persistence_score - model_score:8.3f} km/s"\n      f"  ({(persistence_score - model_score) / persistence_score:.1%})")\nprint("\\n[참고] P1 = 68.408 / P2a = 65.663")\nif model_score >= persistence_score:\n    print("\\n>>> 경고: persistence 미달. 제출하지 마세요.")\n\nfigure, axis = plt.subplots(figsize=(7, 4))\naxis.plot(validation_metrics.horizon_h, validation_metrics.rmse, marker="o", label="model")\naxis.plot(validation_metrics.horizon_h, validation_metrics.persistence_rmse,\n          marker="s", linestyle="--", label="persistence")\naxis.set_xlabel("forecast horizon (h)"); axis.set_ylabel("RMSE (km/s)")\naxis.grid(alpha=0.3); axis.legend()\nplt.tight_layout(); plt.show()\nvalidation_metrics\n\n\n# ===== copied from code_p3_fix.ipynb, cell 22 =====\n\ndel train_loader, val_loader, train_dataset, val_dataset\ngc.collect()\nif DEVICE.type == "cuda":\n    torch.cuda.empty_cache()\n\ntest_dataset = SolarWindDataset(test_image_array, test_image_index, test_inputs,\n                                test_wind, test_wind_valid, test_ch, targets=None)\ntest_loader = make_loader(test_dataset, shuffle=False)\ntest_prediction, predicted_ids = predict(test_loader)\n\nassert predicted_ids == test_inputs.sample_id.tolist()\nassert test_prediction.shape == (len(test_inputs), 12)\nassert np.isfinite(test_prediction).all()\n\nsubmission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)\nsubmission.insert(0, "sample_id", predicted_ids)\nsubmission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)\nshutil.copyfile(checkpoint_path, SUBMISSION_DIR / "model.pth")\n\n# 규정: "code.ipynb 에서 model.pth 를 불러와 추론이 가능해야 함" 을 문자 그대로 충족\nsaved = torch.load(SUBMISSION_DIR / "model.pth", map_location=DEVICE, weights_only=True)\nmodel.load_state_dict(saved["model_state_dict"])\nprint("reloaded from submission/model.pth")\n\nprint("saved:", (SUBMISSION_DIR / "submission.csv").resolve(), submission.shape)\nprint(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))\ndel test_dataset, test_loader\ngc.collect()\nif DEVICE.type == "cuda":\n    torch.cuda.empty_cache()\nsubmission.head()\n\n', P3)

## 2. P9(F3) — 채택된 1 epoch 멤버 재현

In [ ]:
P9 = {}
exec('# ===== copied from code_p9.ipynb, cell 2 =====\n\nfrom pathlib import Path\nimport gc, hashlib, json, math, os, random, shutil, time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nSEED = 777\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\nDATA_ROOT_CANDIDATES = [\n    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,\n    Path("public_dataset/competition_dataset_6h"),\n    Path("/home/jovyan/public_dataset/competition_dataset_6h"),\n    Path("public/public_dataset/competition_dataset_6h"),\n    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),\n    Path("dataset"), Path("/home/jovyan/dataset"),\n]\nDATA_ROOT = None\nfor candidate in DATA_ROOT_CANDIDATES:\n    if candidate is not None and (candidate / "train/inputs.csv").exists():\n        DATA_ROOT = candidate\n        break\nif DATA_ROOT is None:\n    raise FileNotFoundError("데이터 경로 없음")\n\nWORK_DIR = Path("work")\nCACHE_ROOT = WORK_DIR / "cache"\nOUTPUT_DIR = WORK_DIR / "outputs_p9"\nSUBMISSION_DIR = Path("submission")\nfor directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):\n    directory.mkdir(parents=True, exist_ok=True)\n\nIMAGE_COLUMNS = [f"image_{i:02d}" for i in range(20)]\nWIND_COLUMNS = [f"wind_{i:02d}" for i in range(20)]\nTARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]\nHORIZONS = np.arange(1, 13) * 6\nCHANNELS = ("193", "211")\nAU_KM = 1.496e8\nLAST_INDEX = 19.0                  # 윈도우 마지막 관측 시점(T0)의 인덱스\n\n# ---- 코로나홀 추출 (P7 과 동일 — 캐시 그대로 재사용) ----------------------\nCH_CODE_VERSION = "p7a"\nFINE_GRID = (12, 30)\nCH_CUTS = (0.30, 0.45, 0.60)\nBRIGHT_CUT = 1.60\nDISK_MARGIN = 0.95\nPER_FRAME_DISK = True\n\n# ---- 사용할 격자 / 레벨 ---------------------------------------------------\nCH_GRID = (6, 3)\nFOLD_LATITUDE = True\nUSE_LEVELS = ("dark0.45", "bright")\n\n# ---- 탄도 정렬 ------------------------------------------------------------\nREFERENCE_TRANSIT_HOURS = 108.0    # EDA 경험적 최적 지연\nTRANSIT_SPEEDS = (315.0, 345.0, 385.0, 435.0, 500.0, 600.0)   # 고정 모드에서만 사용\nBALLISTIC_OFFSETS = (-4, -2, 0, 2)         # 탄도창 샘플 시점 (6h 스텝)\nGATHER_OFFSETS = (-6, -4, -2, 0, 2, 6)     # 적응형 gather 샘플 시점\nBALLISTIC_SOURCE = "window"        # "window" = 기준 τ ± offset / "speeds" = 속도별 점 샘플\nBALLISTIC_LAT = "profile"          # "profile" = 위도 전부 / "equator" = 적도 1행\nBALLISTIC_LON = "all"              # [R1] "all" = 경도 전부 / "central" = 중앙자오선 1열\nUSE_BALLISTIC_WINDOW = True\n\n# ---- [R2] 적응형 전달 시간 ------------------------------------------------\nADAPTIVE_AREA = True               # 탄도창 인덱스를 샘플별 관측 속도로\nADAPTIVE_GATHER = True             # gather 인덱스를 샘플별 관측 속도로\nUSE_TRANSIT_FLAGS = True           # 관측창 이탈 플래그를 head 에 투입\nSPEED_FLOOR, SPEED_CEIL = 280.0, 800.0\nFLAG_DIM = 4                       # (미래이탈, 과거이탈, 이탈량+, 이탈량-)\n\n# ---- 모델 -----------------------------------------------------------------\nUSE_CH_GATHER = True\nCH_HIDDEN = 64\nCH_BIDIRECTIONAL = True\nGATHER_DIM = 16\nHORIZON_EMBED = 8\nDROPOUT = 0.4\nVERBOSE_MODEL = True\n\n# ---- 학습 -----------------------------------------------------------------\nBATCH_SIZE = 64\nLEARNING_RATE = 3e-4\nWEIGHT_DECAY = 1e-3\nGRAD_CLIP = 1.0\nLOSS_EPSILON = 1e-8\nLOSS_SCALE = 100.0\nCV_EPOCHS = 30\nAUGMENT = True\nAUG_CH_NOISE = 0.05\n\n# ---- 배치 공급 -------------------------------------------------------------\n# 데이터가 전부 메모리 위 numpy 배열이라 I/O 가 없다. torch DataLoader 는\n# train_run 호출마다 워커 프로세스를 새로 띄워서, 이 규모에서는 학습보다 오버헤드가 크다.\n# 전 배열을 DEVICE 에 한 번 올려두고 인덱스 슬라이스로 배치를 만든다.\nBATCHER_LIMIT_MB = 3000    # 이보다 크면 CPU 에 두고 배치마다 옮긴다\n\n# ---- [C][D] 계측기 --------------------------------------------------------\nRUN_FULL_LADDER = False   # 기본은 빠른 A/B 만. 귀속(어느 변경이 효과였나)이 필요할 때 True\nQUICK_SEEDS = (777, 778)  # 대응 비교라 2개로 충분하다 (아래 설명)\nN_FOLDS = 5\nFOLD_MODE = "block"       # [D] "block"=시간 연속 블록 / "balanced"=P7 / "forward"=전진 검증\nCV_SEEDS = (777, 778, 779)     # [C] 전체 사다리용 시드\nLADDER_FOLDS = 3          # 사다리는 앞 3폴드만 (시간 절약). 확정 검증은 전 폴드\nEPOCH_SMOOTH = 3          # [B] epoch 곡선 이동평균 창\nNOISE_SIGMA = 2.0         # [C] 판정 임계 = NOISE_SIGMA * 결합 표준오차\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nUSE_AMP = DEVICE.type == "cuda"\nPIN_MEMORY = DEVICE.type == "cuda"\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\nif DEVICE.type == "cuda":\n    torch.backends.cudnn.benchmark = True\n\nprint("PyTorch:", torch.__version__, "| device:", DEVICE)\nif DEVICE.type == "cuda":\n    print("GPU:", torch.cuda.get_device_name(0))\nprint("data:", DATA_ROOT.resolve())\n\n\n# ===== copied from code_p9.ipynb, cell 4 =====\n\ntrain_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")\ntrain_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")\nval_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")\nval_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")\ntest_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")\nassert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()\nassert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()\n\n\ndef fill_wind(inputs):\n    wind = inputs[WIND_COLUMNS].to_numpy(np.float64)\n    valid = np.isfinite(wind).astype(np.float32)\n    frame = pd.DataFrame(wind).ffill(axis=1).bfill(axis=1)\n    filled = frame.to_numpy(np.float32)\n    return np.nan_to_num(filled, nan=float(np.nanmedian(wind))), valid\n\n\ntrain_wind, train_wind_valid = fill_wind(train_inputs)\nval_wind, val_wind_valid = fill_wind(val_inputs)\ntest_wind, test_wind_valid = fill_wind(test_inputs)\ntrain_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\nval_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\n\n\ndef reconstruct_frame_chains(inputs):\n    """이미지 파일명만으로 프레임 시간축을 복원한다. 행 순서에 의존하지 않는다."""\n    images = inputs[IMAGE_COLUMNS].to_numpy()\n    successor, predecessor, conflicts = {}, {}, 0\n    for row in images:\n        for current, following in zip(row[:-1], row[1:]):\n            if successor.setdefault(current, following) != following:\n                conflicts += 1\n            if predecessor.setdefault(following, current) != current:\n                conflicts += 1\n    names = set(images.ravel().tolist())\n    chains, visited = [], set()\n    for head in sorted(names - set(predecessor)):\n        chain, node = [], head\n        while node is not None and node not in visited:\n            visited.add(node); chain.append(node); node = successor.get(node)\n        chains.append(chain)\n    assert conflicts == 0 and not (names - visited), "사슬 복원 실패"\n    return chains\n\n\ndef sample_chain_index(inputs, chains):\n    position = {name: (c, o)\n                for c, chain in enumerate(chains) for o, name in enumerate(chain)}\n    first = inputs[IMAGE_COLUMNS[0]].to_numpy()\n    chain_id = np.array([position[n][0] for n in first])\n    offset = np.array([position[n][1] for n in first])\n    return chain_id, offset\n\n\nTRAIN_CHAINS = reconstruct_frame_chains(train_inputs)\nVAL_CHAINS = reconstruct_frame_chains(val_inputs)\nTEST_CHAINS = reconstruct_frame_chains(test_inputs)\nTRAIN_CHAIN_ID, TRAIN_OFFSET = sample_chain_index(train_inputs, TRAIN_CHAINS)\n\ntrain_files = [n for chain in TRAIN_CHAINS for n in chain]\nval_files = [n for chain in VAL_CHAINS for n in chain]\ntest_files = [n for chain in TEST_CHAINS for n in chain]\ntrain_map = {n: i for i, n in enumerate(train_files)}\nval_map = {n: i for i, n in enumerate(val_files)}\ntest_map = {n: i for i, n in enumerate(test_files)}\n\nprint(f"train {len(train_inputs):,} 샘플 / 사슬 {len(TRAIN_CHAINS)}개 / 고유 이미지 {len(train_files):,}")\nprint(f"val   {len(val_inputs):,} 샘플 / 사슬 {len(VAL_CHAINS)}개 / 고유 이미지 {len(val_files):,}")\nprint(f"test  {len(test_inputs):,} 샘플 / 사슬 {len(TEST_CHAINS)}개 / 고유 이미지 {len(test_files):,}")\nprint(f"\\ntrain 사슬별 샘플 수: {np.bincount(TRAIN_CHAIN_ID).tolist()}")\n\n# --- [D] 사슬 순서 == 시간 순서 가정 점검 ---------------------------------\nheads = [chain[0] for chain in TRAIN_CHAINS]\nCHAIN_ORDER_IS_TIME = heads == sorted(heads)\nprint(f"\\n[D] 사슬 머리 파일명이 정렬 순서인가: {CHAIN_ORDER_IS_TIME}")\nif not CHAIN_ORDER_IS_TIME:\n    print("    -> 파일명이 시간 인코딩이 아닐 수 있다. FOLD_MODE=\'block\' 의 시간 가정을 확인할 것.")\nprint("    사슬 머리 5개:", heads[:5])\nprint("    사슬 꼬리 5개:", [chain[-1] for chain in TRAIN_CHAINS][-5:])\n\n\n# ===== copied from code_p9.ipynb, cell 6 =====\n\nFINE_LAT, FINE_LON = FINE_GRID\nFINE_CELLS = FINE_LAT * FINE_LON\nN_LEVELS = len(CH_CUTS) + 1\nLEVEL_NAMES = [f"dark{c}" for c in CH_CUTS] + ["bright"]\n\n\ndef load_pair(split, name):\n    planes = []\n    for channel in CHANNELS:\n        with Image.open(DATA_ROOT / split / channel / name) as image:\n            planes.append(np.asarray(image.convert("L"), dtype=np.float32))\n    return np.stack(planes)\n\n\ndef detect_disk(frame):\n    """플레어에 둔감한 원반 검출. 배경과 원반 내부의 중간값을 임계로 쓴다."""\n    plane = frame.mean(axis=0)\n    background = np.percentile(plane, 2.0)\n    interior = np.percentile(plane, 70.0)\n    mask = plane > background + 0.35 * (interior - background)\n    ys, xs = np.nonzero(mask)\n    return float(ys.mean()), float(xs.mean()), float(math.sqrt(mask.sum() / math.pi))\n\n\n_geometry_cache = {}\nGEOMETRY_CACHE_LIMIT = 96\n\n\ndef cell_geometry(side, center_y, center_x, radius):\n    key = (side, round(center_y), round(center_x), round(radius))\n    entry = _geometry_cache.get(key)\n    if entry is None:\n        _, cy, cx, r = key\n        grid_y, grid_x = np.mgrid[0:side, 0:side].astype(np.float32)\n        disk = np.sqrt((grid_y - cy) ** 2 + (grid_x - cx) ** 2) <= r\n        lat = np.clip((grid_y - (cy - r)) / (2 * r) * FINE_LAT, 0, FINE_LAT - 1e-4)\n        lon = np.clip((grid_x - (cx - r)) / (2 * r) * FINE_LON, 0, FINE_LON - 1e-4)\n        cell = (lat.astype(np.int32) * FINE_LON + lon.astype(np.int32))[disk]\n        entry = (disk, cell)\n        _geometry_cache[key] = entry\n        while len(_geometry_cache) > GEOMETRY_CACHE_LIMIT:\n            _geometry_cache.pop(next(iter(_geometry_cache)))\n    return entry\n\n\ndef extract_split(split, filenames):\n    area = np.zeros((len(filenames), N_LEVELS, FINE_CELLS), np.float32)\n    geometry = np.zeros((len(filenames), 4), np.float32)\n    fixed = None\n    if not PER_FRAME_DISK:\n        sample = np.unique(np.linspace(0, len(filenames) - 1, 200).astype(int))\n        measured = np.array([detect_disk(load_pair(split, filenames[i])) for i in sample])\n        fixed = tuple(np.median(measured, axis=0))\n    started = time.perf_counter()\n    for index, name in enumerate(filenames):\n        frame = load_pair(split, name)\n        center_y, center_x, radius = fixed if fixed is not None else detect_disk(frame)\n        disk, cell = cell_geometry(frame.shape[-1], center_y, center_x, radius * DISK_MARGIN)\n        on_disk = frame[:, disk]\n        total = max(on_disk.shape[1], 1)\n        geometry[index] = (center_y, center_x, radius, total)\n        median = np.median(on_disk[:, ::4], axis=1, keepdims=True)\n        normalized = on_disk / np.maximum(median, 1e-3)\n        for level, cut in enumerate(CH_CUTS):\n            selection = np.logical_and(normalized[0] <= cut, normalized[1] <= cut)\n            area[index, level] = np.bincount(cell[selection], minlength=FINE_CELLS) / total\n        selection = np.logical_and(normalized[0] >= BRIGHT_CUT, normalized[1] >= BRIGHT_CUT)\n        area[index, -1] = np.bincount(cell[selection], minlength=FINE_CELLS) / total\n        if (index + 1) % 2000 == 0 or index + 1 == len(filenames):\n            print(f"  {split} {index + 1}/{len(filenames)} "\n                  f"({time.perf_counter() - started:.0f}s)", flush=True)\n    return area, geometry\n\n\ndef cached_extract(split, filenames):\n    tag = (f"{CH_CODE_VERSION}_{FINE_LAT}x{FINE_LON}"\n           f"_c{\'-\'.join(str(c) for c in CH_CUTS)}_b{BRIGHT_CUT}_m{DISK_MARGIN}"\n           f"_{\'perframe\' if PER_FRAME_DISK else \'fixed\'}")\n    area_path = CACHE_ROOT / f"ch_{split}_{tag}.npy"\n    geometry_path = CACHE_ROOT / f"geom_{split}_{tag}.npy"\n    if area_path.exists() and geometry_path.exists():\n        area = np.load(area_path)\n        if area.shape == (len(filenames), N_LEVELS, FINE_CELLS):\n            print(f"캐시 재사용: {area_path.name}")\n            return area, np.load(geometry_path)\n    area, geometry = extract_split(split, filenames)\n    np.save(area_path, area); np.save(geometry_path, geometry)\n    return area, geometry\n\n\ntrain_area, train_geometry = cached_extract("train", train_files)\nval_area, val_geometry = cached_extract("validation", val_files)\ntest_area, test_geometry = cached_extract("test", test_files)\n\nfor name, geometry in [("train", train_geometry), ("val", val_geometry), ("test", test_geometry)]:\n    radius = geometry[:, 2]\n    print(f"{name:5s} 반지름 {radius.mean():.1f} ± {radius.std():.1f}px "\n          f"({radius.std()/radius.mean():.2%}), 원반 픽셀 {geometry[:, 3].mean():,.0f}")\n\n\n# ===== copied from code_p9.ipynb, cell 8 =====\n\ndef aggregate(area, grid_lat, grid_lon, fold):\n    """(n, L, FINE_CELLS) -> (n, L, cells). 면적이 원반 대비 비율이라 단순 합이 정확하다."""\n    block = area.reshape(len(area), N_LEVELS, FINE_LAT, FINE_LON)\n    block = block.reshape(len(area), N_LEVELS, grid_lat, FINE_LAT // grid_lat,\n                          grid_lon, FINE_LON // grid_lon).sum(axis=(3, 5))\n    if fold:\n        block = block + block[:, :, ::-1, :]\n        block = block[:, :, : (grid_lat + 1) // 2, :]\n    return np.ascontiguousarray(block.reshape(len(area), N_LEVELS, -1), dtype=np.float32)\n\n\ndef image_index_matrix(inputs, image_map):\n    return np.asarray([[image_map[n] for n in row]\n                       for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)],\n                      dtype=np.int64)\n\n\ntrain_index_matrix = image_index_matrix(train_inputs, train_map)\nval_index_matrix = image_index_matrix(val_inputs, val_map)\ntest_index_matrix = image_index_matrix(test_inputs, test_map)\n\n# TAU_SCALE 은 train 전용 스칼라. EDA 의 108h 를 train 평균 속도에 고정시킨다.\nTAU_REF_SPEED = float(train_wind.mean())\nTAU_SCALE = REFERENCE_TRANSIT_HOURS / (AU_KM / TAU_REF_SPEED / 3600.0)\nprint(f"[R2] train 평균 속도 {TAU_REF_SPEED:.1f} km/s -> 순수 전달시간 "\n      f"{AU_KM / TAU_REF_SPEED / 3600.0:.1f}h, TAU_SCALE={TAU_SCALE:.3f} "\n      f"(보정 후 {REFERENCE_TRANSIT_HOURS:.0f}h)")\n\n\ndef estimate_tau(wind):\n    """샘플별 전달 지연(시간). (n,)"""\n    speed = np.clip(wind[:, -4:].mean(axis=1), SPEED_FLOOR, SPEED_CEIL)\n    return TAU_SCALE * AU_KM / speed / 3600.0\n\n\ndef reference_index_raw(wind, adaptive):\n    """클리핑 전 기준 인덱스. (n, 12)"""\n    tau = estimate_tau(wind) if adaptive else np.full(len(wind), REFERENCE_TRANSIT_HOURS)\n    return LAST_INDEX + (HORIZONS[None, :] - tau[:, None]) / 6.0\n\n\ndef offset_indices(wind, offsets, adaptive):\n    """기준 인덱스에 오프셋을 더해 클리핑. (n, 12, K)"""\n    raw = reference_index_raw(wind, adaptive)[:, :, None] \\\n        + np.asarray(offsets, np.float64)[None, None, :]\n    return np.clip(raw, 0.0, LAST_INDEX).astype(np.float32)\n\n\ndef fixed_speed_indices(n_samples, speeds):\n    """P3/P7 방식: 고정 속도별 점 샘플. (n, 12, S)"""\n    table = np.zeros((12, len(speeds)), np.float64)\n    for h in range(12):\n        for s, speed in enumerate(speeds):\n            table[h, s] = np.clip(\n                LAST_INDEX + ((h + 1) * 6.0 - AU_KM / speed / 3600.0) / 6.0,\n                0.0, LAST_INDEX)\n    return np.repeat(table[None].astype(np.float32), n_samples, axis=0)\n\n\ndef area_indices(wind):\n    if BALLISTIC_SOURCE == "window":\n        return offset_indices(wind, BALLISTIC_OFFSETS, ADAPTIVE_AREA)\n    return fixed_speed_indices(len(wind), TRANSIT_SPEEDS)\n\n\ndef gather_indices(wind):\n    if ADAPTIVE_GATHER:\n        return offset_indices(wind, GATHER_OFFSETS, True)\n    return fixed_speed_indices(len(wind), TRANSIT_SPEEDS)\n\n\ndef transit_flags(wind):\n    """[R2] 관측창 이탈 플래그. (n, 12, FLAG_DIM)"""\n    raw = reference_index_raw(wind, ADAPTIVE_AREA)\n    over = np.clip(raw - LAST_INDEX, 0.0, None) / 8.0\n    under = np.clip(-raw, 0.0, None) / 8.0\n    return np.stack([(raw > LAST_INDEX).astype(np.float32),\n                     (raw < 0.0).astype(np.float32),\n                     over.astype(np.float32),\n                     under.astype(np.float32)], axis=2).astype(np.float32)\n\n\ndef pick_per_sample(sequence, index):\n    """sequence (n, 20, D) 를 샘플별 실수 인덱스 index (n, K) 에서 선형보간. -> (n, K, D)"""\n    lower = np.floor(index).astype(np.int64)\n    upper = np.minimum(lower + 1, int(LAST_INDEX))\n    weight = (index - lower).astype(np.float32)[:, :, None]\n    rows = np.arange(len(sequence))[:, None]\n    return sequence[rows, lower] * (1.0 - weight) + sequence[rows, upper] * weight\n\n\ndef flatten_ch(grid, indexes):\n    return grid[indexes].reshape(len(indexes), 20, N_USED_LEVELS * N_CELLS).astype(np.float32)\n\n\ndef ballistic_columns():\n    """[R1] 탄도 샘플에 쓸 셀 번호."""\n    lons = list(range(GRID_LON)) if BALLISTIC_LON == "all" else [CENTRAL_LON]\n    return [lat * GRID_LON + lon for lat in AREA_LAT_ROWS for lon in lons], len(lons)\n\n\ndef ballistic_area(grid, indexes, index):\n    """탄도 소스 시각의 코로나홀 면적. index (n, 12, K) -> (n, 12, K*D)"""\n    columns, _ = ballistic_columns()\n    sequence = grid[indexes][:, :, :, columns]\n    sequence = sequence.reshape(len(indexes), 20, -1)\n    n_samples, n_horizon, n_offset = index.shape\n    picked = pick_per_sample(sequence, index.reshape(n_samples, n_horizon * n_offset))\n    return picked.reshape(n_samples, n_horizon, -1).astype(np.float32)\n\n\ndef build_ballistic(grid, indexes, wind):\n    return ballistic_area(grid, indexes, area_indices(wind))\n\n\ndef configure(**overrides):\n    """설정을 바꾸고 파생 상수를 다시 만든다. ablation 은 이 함수로만 한다."""\n    globals().update(overrides)\n    global GRID_LAT, GRID_LON, USED_LAT, N_CELLS, CENTRAL_LON, EQUATOR_ROW\n    global LEVEL_INDEX, N_USED_LEVELS, CH_SEQ_DIM\n    global train_grid, val_grid, test_grid\n    global AREA_LAT_ROWS, N_AREA_OFFSETS, BALLISTIC_DIM, N_GATHER\n\n    GRID_LAT, GRID_LON = CH_GRID\n    assert FINE_LAT % GRID_LAT == 0 and FINE_LON % GRID_LON == 0\n    USED_LAT = (GRID_LAT + 1) // 2 if FOLD_LATITUDE else GRID_LAT\n    N_CELLS = USED_LAT * GRID_LON\n    CENTRAL_LON = GRID_LON // 2\n    EQUATOR_ROW = USED_LAT - 1 if FOLD_LATITUDE else GRID_LAT // 2\n    LEVEL_INDEX = [LEVEL_NAMES.index(name) for name in USE_LEVELS]\n    N_USED_LEVELS = len(LEVEL_INDEX)\n    CH_SEQ_DIM = N_USED_LEVELS * N_CELLS\n\n    train_grid = aggregate(train_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]\n    val_grid = aggregate(val_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]\n    test_grid = aggregate(test_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]\n\n    AREA_LAT_ROWS = list(range(USED_LAT)) if BALLISTIC_LAT == "profile" else [EQUATOR_ROW]\n    N_AREA_OFFSETS = (len(BALLISTIC_OFFSETS) if BALLISTIC_SOURCE == "window"\n                      else len(TRANSIT_SPEEDS))\n    _, n_lons = ballistic_columns()\n    BALLISTIC_DIM = N_AREA_OFFSETS * len(AREA_LAT_ROWS) * n_lons * N_USED_LEVELS\n    N_GATHER = len(GATHER_OFFSETS) if ADAPTIVE_GATHER else len(TRANSIT_SPEEDS)\n\n\nconfigure()\nprint(f"\\n격자 {GRID_LAT}x{GRID_LON} -> 셀 {N_CELLS} (적도 대칭 접기 {FOLD_LATITUDE}), 레벨 {USE_LEVELS}")\nprint(f"ch_seq {CH_SEQ_DIM} / 탄도 {BALLISTIC_DIM} (경도 {BALLISTIC_LON}) / "\n      f"gather {N_GATHER} / 플래그 {FLAG_DIM if USE_TRANSIT_FLAGS else 0}")\n\n_tau = estimate_tau(train_wind)\nprint(f"\\n[R2] 샘플별 전달 지연 분포: {np.percentile(_tau, [5, 25, 50, 75, 95]).round(1)} h "\n      f"(5/25/50/75/95%)  — P7 은 전 샘플 {REFERENCE_TRANSIT_HOURS:.0f}h 고정")\n_raw = reference_index_raw(train_wind, True)\nprint(f"[R2] 관측창 이탈 비율 (idx>19): "\n      f"{[f\'{h}h {(_raw[:, i] > LAST_INDEX).mean():.1%}\' for i, h in enumerate(HORIZONS)][6:]}")\nprint("\\n[R2] 기준 인덱스 분위수 (19 = 마지막 관측)")\nprint(pd.DataFrame(np.percentile(np.clip(_raw, 0, LAST_INDEX), [10, 50, 90], axis=0).T,\n                   index=[f"{h}h" for h in HORIZONS], columns=["p10", "p50", "p90"]).round(2))\n\n\n# ===== copied from code_p9.ipynb, cell 10 =====\n\nSTAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]\nNUM_STATS = len(STAT_NAMES)\n_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5\n_TIME_DENOM = float((_TIME_CENTERED ** 2).sum())\n\n\ndef build_wind_stats(wind):\n    last = wind[:, -1]\n    mean4 = wind[:, -4:].mean(axis=1)\n    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOM\n    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),\n                     wind.max(axis=1), slope, last - mean4,\n                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)\n\n\ndef fit_stats(rows):\n    """주어진 train 행에서만 정규화/복원 통계를 산출한다."""\n    wind = train_wind[rows]\n    targets = train_targets[rows]\n    ch_seq = flatten_ch(train_grid, train_index_matrix[rows]).reshape(-1, CH_SEQ_DIM)\n    ballistic = build_ballistic(train_grid, train_index_matrix[rows], wind\n                                ).reshape(-1, BALLISTIC_DIM)\n    statistics = build_wind_stats(wind)\n    residual = targets - wind[:, -1:]\n    return {\n        "wind_mean": float(wind.mean()), "wind_std": float(wind.std() + 1e-6),\n        "diff_std": float(np.diff(wind, axis=1, prepend=wind[:, :1]).std() + 1e-6),\n        "stats_mean": statistics.mean(axis=0), "stats_std": statistics.std(axis=0) + 1e-6,\n        "ch_mean": ch_seq.mean(axis=0), "ch_std": ch_seq.std(axis=0) + 1e-8,\n        "ballistic_mean": ballistic.mean(axis=0), "ballistic_std": ballistic.std(axis=0) + 1e-8,\n        "residual_mean": residual.mean(axis=0), "residual_std": residual.std(axis=0) + 1e-6,\n        "clip_low": float(targets.min() * 0.95), "clip_high": float(targets.max() * 1.05),\n        "target_mean": targets.mean(axis=0),\n    }\n\n\nBATCH_KEYS = ("wind_seq", "wind_stats", "ch_seq", "ballistic",\n              "gather_idx", "flags", "last_wind")\n\n\ndef build_arrays(inputs, index_matrix, wind, wind_valid, grid, stats, targets=None):\n    """모델 입력 배열 일체를 만든다 (numpy)."""\n    arrays = {\n        "wind_seq": np.stack([\n            (wind - stats["wind_mean"]) / stats["wind_std"],\n            np.diff(wind, axis=1, prepend=wind[:, :1]) / stats["diff_std"],\n            wind_valid], axis=2).astype(np.float32),\n        "wind_stats": ((build_wind_stats(wind) - stats["stats_mean"])\n                       / stats["stats_std"]).astype(np.float32),\n        "ch_seq": ((flatten_ch(grid, index_matrix) - stats["ch_mean"])\n                   / stats["ch_std"]).astype(np.float32),\n        "ballistic": ((build_ballistic(grid, index_matrix, wind) - stats["ballistic_mean"])\n                      / stats["ballistic_std"]).astype(np.float32),\n        "gather_idx": gather_indices(wind),\n        "flags": transit_flags(wind),\n        "last_wind": np.ascontiguousarray(wind[:, -1]).astype(np.float32),\n    }\n    if targets is not None:\n        arrays["target"] = targets.astype(np.float32)\n    arrays["sample_ids"] = inputs.sample_id.to_numpy()\n    return arrays\n\n\nclass Batcher:\n    """배열을 한 번만 텐서로 올려두고 인덱스 슬라이스로 배치를 낸다.\n\n    증강(CH 노이즈)은 배치마다 GPU 에서 더한다 — 표준화된 값이므로 덧셈이 맞다\n    (P7 은 곱셈이라 평균 근처 셀은 노이즈 0, 극단값 셀만 흔들려 방향이 반대였다).\n    """\n\n    def __init__(self, arrays, shuffle=False, seed=SEED, training=False):\n        self.keys = [k for k in BATCH_KEYS + ("target",) if k in arrays]\n        total_mb = sum(arrays[k].nbytes for k in self.keys) / 1024 ** 2\n        self.device = DEVICE if total_mb <= BATCHER_LIMIT_MB else torch.device("cpu")\n        self.tensors = {k: torch.as_tensor(arrays[k]).to(self.device) for k in self.keys}\n        self.sample_ids = arrays["sample_ids"]\n        self.count = len(self.sample_ids)\n        self.shuffle = shuffle\n        self.training = training\n        self.generator = torch.Generator().manual_seed(seed)\n        self.megabytes = total_mb\n\n    def __len__(self):\n        return (self.count + BATCH_SIZE - 1) // BATCH_SIZE\n\n    def __iter__(self):\n        order = (torch.randperm(self.count, generator=self.generator) if self.shuffle\n                 else torch.arange(self.count))\n        order = order.to(self.device)\n        for start in range(0, self.count, BATCH_SIZE):\n            selection = order[start:start + BATCH_SIZE]\n            batch = {k: v.index_select(0, selection) for k, v in self.tensors.items()}\n            if self.training and AUGMENT and AUG_CH_NOISE > 0:\n                for key in ("ch_seq", "ballistic"):\n                    batch[key] = batch[key] + torch.randn(\n                        batch[key].shape, device=self.device) * AUG_CH_NOISE\n            yield batch\n\n    def release(self):\n        self.tensors.clear()\n\n\ndef make_batcher(inputs, index_matrix, wind, wind_valid, grid, stats, targets=None,\n                 shuffle=False, seed=SEED, training=False):\n    arrays = build_arrays(inputs, index_matrix, wind, wind_valid, grid, stats, targets)\n    return Batcher(arrays, shuffle=shuffle, seed=seed, training=training)\n\n\n_probe_stats = fit_stats(np.arange(len(train_inputs)))\n_probe_batcher = make_batcher(train_inputs, train_index_matrix, train_wind, train_wind_valid,\n                              train_grid, _probe_stats, train_targets, shuffle=True)\nprint(f"입력 배열 {_probe_batcher.megabytes:,.0f} MB -> {_probe_batcher.device} 상주, "\n      f"배치 {len(_probe_batcher)}개")\n_probe_batcher.release(); del _probe_batcher; gc.collect()\n\n\n# ===== copied from code_p9.ipynb, cell 12 =====\n\nclass SolarWindP9(nn.Module):\n    def __init__(self, stats):\n        super().__init__()\n        self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)\n        self.stats_encoder = nn.Sequential(\n            nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),\n            nn.Linear(128, 64), nn.SELU(inplace=True))\n        shared_dim = 96 + 64\n\n        self.ch_gru = nn.GRU(CH_SEQ_DIM, CH_HIDDEN, num_layers=2, batch_first=True,\n                             bidirectional=CH_BIDIRECTIONAL)\n        directions = 2 if CH_BIDIRECTIONAL else 1\n        self.ch_dropout = nn.Dropout(DROPOUT)\n        shared_dim += CH_HIDDEN * directions\n\n        head_extra = 0\n        if USE_CH_GATHER:\n            self.gather_project = nn.Sequential(\n                nn.Linear(CH_HIDDEN * directions, GATHER_DIM), nn.ReLU(inplace=True))\n            head_extra += GATHER_DIM * N_GATHER\n        if USE_BALLISTIC_WINDOW:\n            head_extra += BALLISTIC_DIM\n        if USE_TRANSIT_FLAGS:\n            head_extra += FLAG_DIM\n\n        self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)\n        head_input = shared_dim + HORIZON_EMBED + head_extra\n        self.head = nn.Sequential(\n            nn.Linear(head_input, 192), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),\n            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),\n            nn.Linear(96, 1))\n\n        self.register_buffer("residual_mean", torch.as_tensor(stats["residual_mean"]))\n        self.register_buffer("residual_std", torch.as_tensor(stats["residual_std"]))\n        if VERBOSE_MODEL:\n            print(f"shared={shared_dim} head_input={head_input} "\n                  f"(gather {GATHER_DIM * N_GATHER if USE_CH_GATHER else 0}, "\n                  f"ballistic {BALLISTIC_DIM if USE_BALLISTIC_WINDOW else 0}, "\n                  f"flags {FLAG_DIM if USE_TRANSIT_FLAGS else 0})")\n\n    @staticmethod\n    def _gather(projected, index):\n        """projected (B, 20, G) 를 샘플별 실수 인덱스 index (B, H, S) 에서 선형보간."""\n        batch, horizon, offsets = index.shape\n        width = projected.shape[-1]\n        lower = index.floor().clamp(0, LAST_INDEX)\n        weight = (index - lower).unsqueeze(-1)\n        lower = lower.long()\n        upper = (lower + 1).clamp(max=int(LAST_INDEX))\n        flat_lower = lower.reshape(batch, horizon * offsets, 1).expand(-1, -1, width)\n        flat_upper = upper.reshape(batch, horizon * offsets, 1).expand(-1, -1, width)\n        low = torch.gather(projected, 1, flat_lower).reshape(batch, horizon, offsets, width)\n        high = torch.gather(projected, 1, flat_upper).reshape(batch, horizon, offsets, width)\n        return low * (1.0 - weight) + high * weight\n\n    def forward(self, wind_seq, wind_stats, ch_seq, ballistic, gather_idx, flags):\n        _, wind_hidden = self.wind_gru(wind_seq)\n        ch_sequence, ch_hidden = self.ch_gru(ch_seq)\n        if CH_BIDIRECTIONAL:\n            ch_last = torch.cat([ch_hidden[-2], ch_hidden[-1]], dim=1)\n        else:\n            ch_last = ch_hidden[-1]\n        shared = torch.cat([F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats),\n                            self.ch_dropout(F.relu(ch_last))], dim=1)\n\n        batch = shared.shape[0]\n        head_parts = [shared.unsqueeze(1).expand(batch, 12, shared.shape[1]),\n                      self.horizon_embedding.unsqueeze(0).expand(batch, 12, HORIZON_EMBED)]\n        if USE_CH_GATHER:\n            projected = self.gather_project(ch_sequence)\n            head_parts.append(self._gather(projected, gather_idx).flatten(2))\n        if USE_BALLISTIC_WINDOW:\n            head_parts.append(ballistic)\n        if USE_TRANSIT_FLAGS:\n            head_parts.append(flags)\n        z = self.head(torch.cat(head_parts, dim=2)).squeeze(-1)\n        return z * self.residual_std + self.residual_mean\n\n\ndef build_model(stats):\n    return SolarWindP9(stats).to(DEVICE)\n\n\n_probe = build_model(_probe_stats)\nprint("trainable parameters:",\n      f"{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}")\ndel _probe; gc.collect()\n\n\n# ===== copied from code_p9.ipynb, cell 14 =====\n\ndef official_rmse(y_true, y_pred):\n    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))\n    return float(per_horizon.mean()), per_horizon\n\n\ndef metric_loss(prediction, target):\n    error = (prediction - target) / LOSS_SCALE\n    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()\n\n\n@torch.no_grad()\ndef predict_with(model, batcher, clip_low, clip_high):\n    model.eval()\n    predictions = []\n    for batch in batcher:\n        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY) for k in BATCH_KEYS}\n        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n            residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],\n                             moved["ballistic"], moved["gather_idx"], moved["flags"])\n        prediction = (residual.float() + moved["last_wind"].unsqueeze(1)\n                      ).clamp(clip_low, clip_high)\n        predictions.append(prediction.cpu().numpy())\n    return np.concatenate(predictions).astype(np.float64), list(batcher.sample_ids)\n\n\nTRAIN_CLIMATOLOGY = train_targets.mean(axis=0)\n\n\ndef baselines(targets, wind):\n    persistence = np.repeat(wind[:, -1:], 12, axis=1)\n    climatology = np.tile(TRAIN_CLIMATOLOGY, (len(targets), 1))\n    return (np.sqrt(((persistence - targets) ** 2).mean(axis=0)),\n            np.sqrt(((climatology - targets) ** 2).mean(axis=0)))\n\n\nVAL_PERSISTENCE, VAL_CLIMATOLOGY = baselines(val_targets, val_wind)\nprint(f"validation  persistence {VAL_PERSISTENCE.mean():.3f} / "\n      f"climatology {VAL_CLIMATOLOGY.mean():.3f} km/s")\nprint(pd.DataFrame({"horizon": HORIZONS,\n                    "persistence": VAL_PERSISTENCE.round(2),\n                    "climatology": VAL_CLIMATOLOGY.round(2)}).to_string(index=False))\n\n\n# ===== copied from code_p9.ipynb, cell 16 =====\n\ndef build_folds(mode=None, n_folds=N_FOLDS):\n    mode = mode or FOLD_MODE\n    n_chains = len(TRAIN_CHAINS)\n    all_rows = np.arange(len(train_inputs))\n\n    if mode == "balanced":                      # P7 방식 (비교용)\n        counts = np.bincount(TRAIN_CHAIN_ID, minlength=n_chains)\n        order = np.argsort(counts)[::-1]\n        assignment = np.zeros(n_chains, np.int64)\n        loads = np.zeros(n_folds, np.int64)\n        for chain in order:\n            fold = int(np.argmin(loads))\n            assignment[chain] = fold\n            loads[fold] += counts[chain]\n        return [(np.flatnonzero(assignment[TRAIN_CHAIN_ID] != f),\n                 np.flatnonzero(assignment[TRAIN_CHAIN_ID] == f)) for f in range(n_folds)]\n\n    edges = np.linspace(0, n_chains, n_folds + 1).astype(int)\n    folds = []\n    for f in range(n_folds):\n        block = np.arange(edges[f], edges[f + 1])\n        evaluate = np.flatnonzero(np.isin(TRAIN_CHAIN_ID, block))\n        if mode == "forward":\n            train = np.flatnonzero(np.isin(TRAIN_CHAIN_ID, np.arange(0, edges[f])))\n        else:\n            train = np.setdiff1d(all_rows, evaluate)\n        if len(train) == 0 or len(evaluate) == 0:\n            continue\n        folds.append((train, evaluate))\n    return folds\n\n\nFOLDS = build_folds()\nprint(f"[D] FOLD_MODE={FOLD_MODE} / 폴드 {len(FOLDS)}개")\nprint("    (학습, 평가) 샘플 수:", [(len(a), len(b)) for a, b in FOLDS])\nprint("    참고 balanced:", [(len(a), len(b)) for a, b in build_folds(\'balanced\')])\n\n\ndef train_run(train_rows, evaluate_rows, seed=SEED, epochs=CV_EPOCHS, verbose=False):\n    """고정 epoch 학습. 매 epoch 의 홀드아웃 horizon별 RMSE 를 기록만 한다."""\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n    stats = fit_stats(train_rows)\n    train_loader = make_batcher(\n        train_inputs.iloc[train_rows], train_index_matrix[train_rows], train_wind[train_rows],\n        train_wind_valid[train_rows], train_grid, stats, train_targets[train_rows],\n        shuffle=True, seed=seed, training=True)\n    evaluate_loader = make_batcher(\n        train_inputs.iloc[evaluate_rows], train_index_matrix[evaluate_rows],\n        train_wind[evaluate_rows], train_wind_valid[evaluate_rows], train_grid, stats,\n        train_targets[evaluate_rows], shuffle=False, seed=seed)\n    evaluate_targets = train_targets[evaluate_rows]\n\n    model = build_model(stats)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,\n                                  weight_decay=WEIGHT_DECAY)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)\n\n    curve = np.zeros((epochs, 12), np.float64)\n    for epoch in range(epochs):\n        model.train()\n        for batch in train_loader:\n            moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)\n                     for k in BATCH_KEYS + ("target",)}\n            optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n                residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],\n                                 moved["ballistic"], moved["gather_idx"], moved["flags"])\n            loss = metric_loss(residual.float() + moved["last_wind"].unsqueeze(1),\n                               moved["target"])\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n            scaler.step(optimizer); scaler.update()\n        scheduler.step()\n        prediction, _ = predict_with(model, evaluate_loader,\n                                     stats["clip_low"], stats["clip_high"])\n        curve[epoch] = official_rmse(evaluate_targets, prediction)[1]\n        if verbose:\n            print(f"    epoch {epoch + 1:03d} rmse {curve[epoch].mean():7.3f} "\n                  f"(72h {curve[epoch][-1]:6.2f})", flush=True)\n    train_loader.release(); evaluate_loader.release()\n    del model, train_loader, evaluate_loader\n    gc.collect()\n    if DEVICE.type == "cuda":\n        torch.cuda.empty_cache()\n    return curve\n\n\ndef smooth_curve(curve, window=EPOCH_SMOOTH):\n    """[B] epoch 축 이동평균. argmin 이 단발 노이즈를 집는 것을 막는다."""\n    if window <= 1:\n        return curve\n    left = window // 2\n    padded = np.pad(curve, ((left, window - 1 - left), (0, 0)), mode="edge")\n    return np.stack([padded[i:i + window].mean(axis=0) for i in range(len(curve))])\n\n\ndef settings_signature(settings):\n    """설정 해시. 캐시 키에 넣어야 설정을 바꿨을 때 옛 결과를 조용히 재사용하지 않는다."""\n    payload = json.dumps({k: str(v) for k, v in sorted(settings.items())})\n    payload += f"|{FOLD_MODE}"\n    return hashlib.md5(payload.encode()).hexdigest()[:8]\n\n\ndef run_cv(label, folds, seeds=CV_SEEDS, epochs=CV_EPOCHS, resume=True, signature=""):\n    """-> curves (n_folds, n_seeds, epochs, 12). 칸 단위로 저장하고 재시작 시 건너뛴다."""\n    path = OUTPUT_DIR / f"cv_{label}{\'_\' + signature if signature else \'\'}.npy"\n    shape = (len(folds), len(seeds), epochs, 12)\n    if resume and path.exists():\n        cached = np.load(path)\n        if cached.shape == shape:\n            print(f"  [{label}] 캐시 재사용 {path.name} "\n                  f"(다시 돌리려면 파일을 지우거나 resume=False)", flush=True)\n            return cached\n        print(f"  [{label}] 캐시 형상 불일치 {cached.shape} != {shape} -> 재계산")\n\n    started = time.perf_counter()\n    curves = np.zeros(shape)\n    total = len(folds) * len(seeds)\n    for f, (train_rows, evaluate_rows) in enumerate(folds):\n        for s, seed in enumerate(seeds):\n            curves[f, s] = train_run(train_rows, evaluate_rows, seed, epochs)\n            done = f * len(seeds) + s + 1\n            elapsed = time.perf_counter() - started\n            print(f"  [{label}] fold {f + 1}/{len(folds)} seed {seed} "\n                  f"best {curves[f, s].mean(axis=1).min():7.3f} | "\n                  f"{done}/{total} runs, {elapsed / 60:.1f}분 경과, "\n                  f"남은 {elapsed / done * (total - done) / 60:.1f}분", flush=True)\n    np.save(path, curves)\n    print(f"  [{label}] 완료 ({(time.perf_counter() - started) / 60:.1f}분) -> {path.name}",\n          flush=True)\n    return curves\n\n\ndef paired_scores(curves, epoch):\n    """(fold, seed) 조합별 평균 RMSE. -> (F*S,), (F*S, 12)"""\n    smoothed = np.stack([[smooth_curve(curves[f, s]) for s in range(curves.shape[1])]\n                         for f in range(curves.shape[0])])\n    per_horizon = smoothed[:, :, epoch].reshape(-1, 12)\n    return per_horizon.mean(axis=1), per_horizon\n\n\ndef paired_delta(baseline, variant, epoch, columns=slice(None)):\n    """**같은 (fold, seed) 쌍**에서의 차이를 본다.\n\n    두 config 은 폴드와 시드를 공유한다. 독립 표본처럼 비교하면 공통 분산(그 폴드가\n    원래 어려웠다, 그 시드가 원래 나빴다)이 오차에 그대로 남는다. 짝을 지어 빼면\n    그 성분이 상쇄되고, 같은 판정력을 훨씬 적은 실행으로 얻는다.\n    """\n    _, base_h = paired_scores(baseline, epoch)\n    _, var_h = paired_scores(variant, epoch)\n    difference = var_h[:, columns].mean(axis=1) - base_h[:, columns].mean(axis=1)\n    n = len(difference)\n    se = float(difference.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan\n    return {"delta": float(difference.mean()), "se": se, "n": n,\n            "per_pair": difference,\n            "unpaired_se": float(np.hypot(base_h[:, columns].mean(axis=1).std(ddof=1),\n                                          var_h[:, columns].mean(axis=1).std(ddof=1))\n                                 / np.sqrt(n)) if n > 1 else np.nan}\n\n\ndef verdict(delta, se, sigma=NOISE_SIGMA):\n    if not np.isfinite(se):\n        return "판정불가"\n    if delta + sigma * se < 0:\n        return "개선"\n    if delta - sigma * se > 0:\n        return "악화"\n    return "차이없음"\n\n\ndef score_at(curves, epoch):\n    """(fold, seed) 별 평균 RMSE 와 horizon별 평균. [C] 결합 표준오차."""\n    smoothed = np.stack([[smooth_curve(curves[f, s]) for s in range(curves.shape[1])]\n                         for f in range(curves.shape[0])])          # (F, S, E, 12)\n    per_horizon = smoothed[:, :, epoch].reshape(-1, 12)             # (F*S, 12)\n    runs = per_horizon.mean(axis=1)\n    fold_means = smoothed[:, :, epoch].mean(axis=(1, 2))            # 폴드별\n    seed_means = smoothed[:, :, epoch].mean(axis=(0, 2))            # 시드별\n    return {"per_horizon": per_horizon.mean(axis=0),\n            "cv": float(runs.mean()),\n            "se": float(runs.std(ddof=1) / np.sqrt(len(runs))) if len(runs) > 1 else np.nan,\n            "fold_sd": float(fold_means.std(ddof=1)) if len(fold_means) > 1 else np.nan,\n            "seed_sd": float(seed_means.std(ddof=1)) if len(seed_means) > 1 else np.nan,\n            "smoothed_mean": smoothed.mean(axis=(0, 1))}\n\n\n\n# Published P9 member = adopted P7/F3 configuration, trained for its selected\n# one epoch.  Do not switch this to G3: P9\'s own paired CV chose F3.\nP9_SETTINGS = dict(\n    CH_GRID=(6, 3), FOLD_LATITUDE=True,\n    USE_LEVELS=("dark0.45", "bright"),\n    TRANSIT_SPEEDS=(315.0, 345.0, 385.0, 435.0, 500.0, 600.0),\n    BALLISTIC_SOURCE="window", BALLISTIC_LAT="profile", BALLISTIC_LON="all",\n    BALLISTIC_OFFSETS=(-4, -2, 0, 2),\n    ADAPTIVE_AREA=False, ADAPTIVE_GATHER=False, USE_TRANSIT_FLAGS=False,\n    USE_CH_GATHER=True, USE_BALLISTIC_WINDOW=True, CH_BIDIRECTIONAL=True,\n)\nP9_FINAL_EPOCHS = 1\nconfigure(**P9_SETTINGS)\nFINAL_STATS = fit_stats(np.arange(len(train_inputs)))\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\nfinal_loader = make_batcher(\n    train_inputs, train_index_matrix, train_wind, train_wind_valid,\n    train_grid, FINAL_STATS, train_targets, shuffle=True, seed=SEED, training=True,\n)\nmodel = build_model(FINAL_STATS)\noptimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)\nscheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CV_EPOCHS)\nscaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)\nfor epoch in range(P9_FINAL_EPOCHS):\n    model.train()\n    for batch in final_loader:\n        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)\n                 for k in BATCH_KEYS + ("target",)}\n        optimizer.zero_grad(set_to_none=True)\n        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n            residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],\n                             moved["ballistic"], moved["gather_idx"], moved["flags"])\n        prediction = residual.float() + moved["last_wind"].unsqueeze(1)\n        loss = metric_loss(prediction, moved["target"])\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n        scaler.step(optimizer); scaler.update()\n    scheduler.step()\n    print(f"P9/F3 epoch {epoch + 1}/{P9_FINAL_EPOCHS} complete", flush=True)\nfinal_loader.release()\n\ndef p9_predict(inputs, index_matrix, wind, valid, grid, targets=None):\n    batcher = make_batcher(inputs, index_matrix, wind, valid, grid, FINAL_STATS,\n                           targets, shuffle=False)\n    prediction, ids = predict_with(model, batcher, FINAL_STATS["clip_low"],\n                                   FINAL_STATS["clip_high"])\n    batcher.release()\n    return prediction, ids\n\nval_prediction, val_ids = p9_predict(\n    val_inputs, val_index_matrix, val_wind, val_wind_valid, val_grid, val_targets,\n)\ntest_prediction, test_ids = p9_predict(\n    test_inputs, test_index_matrix, test_wind, test_wind_valid, test_grid,\n)\nassert val_ids == val_inputs.sample_id.tolist()\nassert test_ids == test_inputs.sample_id.tolist()\n', P9)

## 3. N1 — 원래 SpeedNet best-checkpoint 레시피 재현

In [ ]:
N1 = {}
exec('# ===== copied from code_N1.ipynb, cell 1 =====\n\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n# Python, NumPy, PyTorch의 seed를 같게 설정\nSEED = 777\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\n# notebook과 같은 위치의 기본 데이터/출력 경로를 사용합니다.\nDATA_ROOT = Path("public_dataset/competition_dataset_6h")\nOUTPUT_DIR = Path("outputs/speednet_n1")\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n# 전처리 cache는 read-only일 수 있는 DATA_ROOT 대신 작업 경로 아래에 만듭니다.\nCACHE_ROOT = Path("cache/speednet_n1")\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nSUBMISSION_DIR = Path("submission")\nSUBMISSION_DIR.mkdir(parents=True, exist_ok=True)\n\n# --- 전체 실행 시간 예산 ---------------------------------------------------\n# 20분 안에 submission.csv까지 나와야 하므로 학습 루프가 벽시계 시간을 보고 스스로 멈춥니다.\nRUN_STARTED = time.perf_counter()\nTOTAL_BUDGET_SECONDS = 20 * 60\nINFERENCE_RESERVE_SECONDS = 150   # test 추론 + submission 저장용으로 남겨둘 시간\n\n\ndef elapsed_seconds():\n    return time.perf_counter() - RUN_STARTED\n\n\ndef remaining_seconds():\n    return TOTAL_BUDGET_SECONDS - elapsed_seconds()\n\n\nIMAGE_SIZE = 64\nCHANNELS = ("193", "211")\n# 모델 입력 채널: 논문 Eq.(2)의 3채널 EUV 맵 자리에 [193, 211, CH 이진맵]을 넣습니다.\nMAP_CHANNELS = ("193", "211", "ch_binary")\n\n# --- 논문 2.1절 / Fig. 4 전처리 파라미터 -----------------------------------\nPROJECTION_LON_LIMIT = 75.0     # Fig. 4d bounding box: 중심자오선 기준 경도 범위\nPROJECTION_LAT_LIMIT = 75.0     # Fig. 4d bounding box: 위도 범위\nPROJECTION_SUPERSAMPLE = 4      # 투영 격자 anti-aliasing 배율\nLIMB_MARGIN = 0.97              # Fig. 4b off-limb 제거: 원반 반지름의 97% 안쪽만 유효\nCH_AREA_TARGET = 0.10           # 동적 임계값 보정 기준(원반 평균 CH 면적 비율)\nCENTRAL_MERIDIAN_BANDS = (10.0, 30.0)   # Fig. 4f 빨간 선 = 중심자오선 +-10도\nLOW_LATITUDE_LIMIT = 30.0       # 저위도 CH가 HSS의 주 원인 (논문 1절)\nGEOMETRY_SAMPLE = 64            # 태양 원반 중심/반지름 추정에 쓸 이미지 수\nTHRESHOLD_SAMPLE = 256          # 동적 임계값 보정에 쓸 이미지 수\nCACHE_WORKERS = min(16, (os.cpu_count() or 4))\n\n# --- 논문 Table 4 하이퍼파라미터 -------------------------------------------\nPAPER_HPARAMS = {\n    "conv_blocks": 4,\n    "conv_channels": (16, 32, 64, 128),\n    "lstm_units": 100,\n    "ffnn_units": {"SpeedNet-EUV": 200, "SpeedNet-BM": 400},\n    "batch_size": 12,\n    "learning_rate": 0.0001,\n    "epochs": 100,\n    "optimizer": "Adam",\n    "loss": "MSE",\n    "model_callback": "min validation loss",\n    "early_stopping_patience": 20,\n}\n\nCONV_CHANNELS = PAPER_HPARAMS["conv_channels"]\nLSTM_UNITS = PAPER_HPARAMS["lstm_units"]\nFFNN_UNITS = PAPER_HPARAMS["ffnn_units"]["SpeedNet-EUV"]   # EUV 맵 입력이므로 200\nLEARNING_RATE = PAPER_HPARAMS["learning_rate"]\nEPOCHS = PAPER_HPARAMS["epochs"]\nEARLY_STOPPING_PATIENCE = PAPER_HPARAMS["early_stopping_patience"]\n\n# 논문 batch=12는 학습 샘플 1,060개 기준입니다. 대회 학습 데이터는 9,607개라\n# 그대로 두면 epoch당 801 step이 되어 20분 예산 안에 학습이 끝나지 않습니다.\nPAPER_EXACT_BATCH_SIZE = False\nBATCH_SIZE = PAPER_HPARAMS["batch_size"] if PAPER_EXACT_BATCH_SIZE else 64\n\nNUM_WORKERS = min(8, (os.cpu_count() or 4))\n# CUDA를 사용할 수 있으면 GPU와 mixed precision(AMP)을 자동으로 사용합니다.\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nUSE_AMP = DEVICE.type == "cuda"\nPIN_MEMORY = DEVICE.type == "cuda"\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\nif DEVICE.type == "cuda":\n    torch.backends.cudnn.benchmark = True\nelse:\n    print("WARNING: CUDA Unavailable")\n\nprint("PyTorch:", torch.__version__)\nprint("device:", DEVICE)\nif DEVICE.type == "cuda":\n    print("GPU:", torch.cuda.get_device_name(0))\nprint("data:", DATA_ROOT.resolve())\nprint("batch size:", BATCH_SIZE, "(paper:", PAPER_HPARAMS["batch_size"], ")")\n\n\n\n# ===== copied from code_N1.ipynb, cell 3 =====\n\nIMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]\nWIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]\nTARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]\n\n# 세 split은 미리 분리되어 있으며 notebook에서 다시 나누지 않습니다.\ninfo = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))\ntrain_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")\ntrain_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")\nval_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")\nval_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")\ntest_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")\ntest_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")\n\n# 입력과 정답의 sample 순서, ID 중복, split 간 누락/혼입을 학습 전에 검사합니다.\nassert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()\nassert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()\nassert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()\nassert train_inputs.sample_id.is_unique\nassert val_inputs.sample_id.is_unique\nassert test_inputs.sample_id.is_unique\nassert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)\nassert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)\nassert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)\nassert set(IMAGE_COLUMNS + WIND_COLUMNS).issubset(train_inputs.columns)\nassert set(IMAGE_COLUMNS + WIND_COLUMNS).issubset(val_inputs.columns)\nassert set(TARGET_COLUMNS).issubset(train_targets_frame.columns)\nassert set(TARGET_COLUMNS).issubset(val_targets_frame.columns)\nassert not any(column.startswith("target_") for column in test_inputs.columns)\n\n# 모델 입력에 바로 사용할 수 있도록 target과 row index를 NumPy 배열로 준비합니다.\ntrain_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\nval_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)\ntrain_index = np.arange(len(train_inputs), dtype=np.int64)\nval_index = np.arange(len(val_inputs), dtype=np.int64)\ntest_index = np.arange(len(test_inputs), dtype=np.int64)\n\nprint(json.dumps(info["counts"], ensure_ascii=False, indent=2))\nprint("train/validation/test samples:", len(train_index), len(val_index), len(test_index))\n\n\n\n# ===== copied from code_N1.ipynb, cell 5 =====\n\ndef open_gray(path):\n    with Image.open(path) as image:\n        return np.asarray(image.convert("L"), dtype=np.float32)\n\n\ndef unique_filenames(inputs):\n    # 여러 sample에서 반복되는 PNG는 고유 파일당 한 번만 전처리합니다.\n    return sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())\n\n\ndef subsample(items, count):\n    step = max(1, len(items) // count)\n    return items[::step][:count]\n\n\ndef estimate_disk_geometry(split, filenames, channel):\n    # 논문 Fig. 4b: off-limb 픽셀을 지우려면 태양 원반의 중심과 반지름이 필요합니다.\n    # FITS 헤더(RSUN_OBS, CRPIX)가 없으므로 train 영상 평균에서 직접 추정합니다.\n    image_root = DATA_ROOT / split / channel\n    picked = subsample(filenames, GEOMETRY_SAMPLE)\n    accumulator = None\n    for filename in picked:\n        frame = open_gray(image_root / filename)\n        accumulator = frame if accumulator is None else accumulator + frame\n    mean_image = accumulator / len(picked)\n    height, width = mean_image.shape\n\n    # 밝은 원반 영역의 무게중심을 원반 중심으로 사용합니다.\n    bright = mean_image > 0.2 * float(np.percentile(mean_image, 99.5))\n    rows, columns = np.nonzero(bright)\n    center_y = float(rows.mean())\n    center_x = float(columns.mean())\n\n    # 중심에서의 반경별 평균 밝기가 원반 내부 밝기의 35% 아래로 떨어지는 지점을 limb으로 봅니다.\n    grid_y, grid_x = np.ogrid[:height, :width]\n    radius_map = np.hypot(grid_y - center_y, grid_x - center_x)\n    max_radius = int(min(center_x, center_y, width - center_x, height - center_y))\n    bins = np.clip(radius_map.astype(np.int64), 0, max_radius)\n    total = np.bincount(bins.ravel(), weights=mean_image.ravel(), minlength=max_radius + 1)\n    counts = np.bincount(bins.ravel(), minlength=max_radius + 1)\n    profile = total / np.maximum(counts, 1)\n    interior = float(profile[: max(4, max_radius // 4)].mean())\n    inside = np.nonzero(profile > 0.35 * interior)[0]\n    radius = float(inside[-1]) if len(inside) else 0.40 * width\n    radius = float(np.clip(radius, 0.20 * width, 0.50 * width)) * 0.995\n\n    geometry = {\n        "channel": channel,\n        "height": int(height),\n        "width": int(width),\n        "center_x": center_x,\n        "center_y": center_y,\n        "radius": radius,\n    }\n    print(f"disk geometry [{channel}]: {width}x{height} "\n          f"center=({center_x:.1f},{center_y:.1f}) radius={radius:.1f}px")\n    return geometry\n\n\ndef build_projection_grid(geometry, size=IMAGE_SIZE, supersample=PROJECTION_SUPERSAMPLE):\n    # Stonyhurst 헬리오그래픽 좌표(경도 x, 위도 y) 격자를 원반 이미지 좌표로 되돌리는 사상.\n    # B0(태양 자전축 기울기)은 연중 +-7.25도로 작고 헤더가 없으므로 0으로 둡니다.\n    fine = size * supersample\n    longitude = np.deg2rad(\n        np.linspace(-PROJECTION_LON_LIMIT, PROJECTION_LON_LIMIT, fine, dtype=np.float64)\n    )\n    latitude = np.deg2rad(\n        np.linspace(PROJECTION_LAT_LIMIT, -PROJECTION_LAT_LIMIT, fine, dtype=np.float64)\n    )\n    longitude_grid, latitude_grid = np.meshgrid(longitude, latitude)\n    # 구면 위 점을 관측자 시선 방향으로 정사영한 좌표 (단위: 태양 반지름)\n    solar_x = np.cos(latitude_grid) * np.sin(longitude_grid)\n    solar_y = np.sin(latitude_grid)\n    pixel_x = geometry["center_x"] + solar_x * geometry["radius"]\n    pixel_y = geometry["center_y"] - solar_y * geometry["radius"]\n\n    # bilinear 보간 가중치를 미리 만들어 두고 이미지마다 fancy indexing만 수행합니다.\n    left = np.clip(np.floor(pixel_x), 0, geometry["width"] - 2).astype(np.int64)\n    top = np.clip(np.floor(pixel_y), 0, geometry["height"] - 2).astype(np.int64)\n    return {\n        "size": size,\n        "supersample": supersample,\n        "left": left,\n        "top": top,\n        "weight_x": (pixel_x - left).astype(np.float32),\n        "weight_y": (pixel_y - top).astype(np.float32),\n    }\n\n\ndef project(frame, grid):\n    left, top = grid["left"], grid["top"]\n    weight_x, weight_y = grid["weight_x"], grid["weight_y"]\n    upper = frame[top, left] * (1.0 - weight_x) + frame[top, left + 1] * weight_x\n    lower = frame[top + 1, left] * (1.0 - weight_x) + frame[top + 1, left + 1] * weight_x\n    fine = upper * (1.0 - weight_y) + lower * weight_y\n    size, supersample = grid["size"], grid["supersample"]\n    # supersample 격자를 box 평균으로 줄여 aliasing을 없앱니다.\n    return fine.reshape(size, supersample, size, supersample).mean(axis=(1, 3))\n\n\n# 투영 격자 위의 위도/경도와, 면적 가중치(구면 면적소는 cos(위도)에 비례)\nGRID_LATITUDE = np.linspace(PROJECTION_LAT_LIMIT, -PROJECTION_LAT_LIMIT, IMAGE_SIZE)\nGRID_LONGITUDE = np.linspace(-PROJECTION_LON_LIMIT, PROJECTION_LON_LIMIT, IMAGE_SIZE)\n\n\n\ndef build_on_disk_mask(size=IMAGE_SIZE, supersample=PROJECTION_SUPERSAMPLE):\n    # 논문 Fig. 4b의 off-limb 제거. +-75도 격자의 모서리는 원반 가장자리 바깥을 찍게 되므로\n    # 정사영 거리가 limb의 LIMB_MARGIN을 넘는 격자점은 "관측값 없음"으로 버립니다.\n    # (그대로 두면 원반 밖 어두운 코로나가 코로나 홀로 오검출됩니다.)\n    # supersample 부분 픽셀이 섞이면 경계에 어두운 띠가 생기므로,\n    # 16개 subsample이 모두 원반 안인 격자점만 유효로 봅니다.\n    fine = size * supersample\n    longitude, latitude = np.meshgrid(\n        np.deg2rad(np.linspace(-PROJECTION_LON_LIMIT, PROJECTION_LON_LIMIT, fine)),\n        np.deg2rad(np.linspace(PROJECTION_LAT_LIMIT, -PROJECTION_LAT_LIMIT, fine)),\n    )\n    inside = (\n        (np.cos(latitude) * np.sin(longitude)) ** 2 + np.sin(latitude) ** 2\n    ) <= LIMB_MARGIN ** 2\n    coverage = inside.reshape(size, supersample, size, supersample).mean(axis=(1, 3))\n    return coverage >= 1.0 - 1e-9\n\n\nON_DISK = build_on_disk_mask()\nON_DISK_FLOAT = ON_DISK.astype(np.float32)\nAREA_WEIGHT = (\n    np.broadcast_to(\n        np.cos(np.deg2rad(GRID_LATITUDE))[:, None].astype(np.float32),\n        (IMAGE_SIZE, IMAGE_SIZE),\n    )\n    * ON_DISK_FLOAT\n)\nMERIDIAN_MASKS = [np.abs(GRID_LONGITUDE) <= band for band in CENTRAL_MERIDIAN_BANDS]\nLOW_LATITUDE_MASK = np.abs(GRID_LATITUDE) <= LOW_LATITUDE_LIMIT\nCH_FEATURE_NAMES = (\n    "ch_area_full_disk",\n    f"ch_area_meridian_{int(CENTRAL_MERIDIAN_BANDS[0])}deg",\n    f"ch_area_meridian_{int(CENTRAL_MERIDIAN_BANDS[1])}deg",\n    f"ch_area_low_latitude_{int(LOW_LATITUDE_LIMIT)}deg",\n)\nCH_FEATURE_DIM = len(CH_FEATURE_NAMES)\n\n\ndef coronal_hole_features(binary):\n    # 논문 Fig. 4f: 중심자오선 +-10도를 지나는 CH가 3~4일 뒤 지구 HSS의 주 원인.\n    features = [float((binary * AREA_WEIGHT).sum() / AREA_WEIGHT.sum())]\n    for mask in MERIDIAN_MASKS:\n        weight = AREA_WEIGHT[:, mask]\n        features.append(float((binary[:, mask] * weight).sum() / weight.sum()))\n    weight = AREA_WEIGHT[LOW_LATITUDE_MASK]\n    features.append(float((binary[LOW_LATITUDE_MASK] * weight).sum() / weight.sum()))\n    return np.asarray(features, dtype=np.float32)\n\n\ndef calibrate_ch_threshold(split, filenames, grid):\n    # Milosic et al. (2023)의 동적 임계화: 임계값을 이미지 중앙값에 비례시키고,\n    # 비례상수는 train 표본에서 평균 CH 면적이 CH_AREA_TARGET이 되도록 한 번만 맞춥니다.\n    image_root = DATA_ROOT / split / CHANNELS[0]\n    ratios = []\n    for filename in subsample(filenames, THRESHOLD_SAMPLE):\n        projected = project(open_gray(image_root / filename), grid)[ON_DISK]\n        median = float(np.median(projected))\n        if median > 0:\n            ratios.append(projected / median)\n    pooled = np.concatenate(ratios)\n    ch_ratio = float(np.quantile(pooled, CH_AREA_TARGET))\n    print(f"on-disk grid points: {int(ON_DISK.sum())}/{ON_DISK.size} "\n          f"({ON_DISK.mean():.1%} of the {IMAGE_SIZE}x{IMAGE_SIZE} map)")\n    print(f"coronal hole dynamic threshold: I < {ch_ratio:.4f} x median(I)  "\n          f"(target area {CH_AREA_TARGET:.0%}, {len(ratios)} images)")\n    return ch_ratio\n\n\n\n# ===== copied from code_N1.ipynb, cell 6 =====\n\ndef prepare_speednet_cache(split, inputs, grids, ch_ratio):\n    # baseline의 resize cache와 같은 방식이지만, 저장하는 내용이\n    # [193 투영맵, 211 투영맵, CH 이진맵]과 CH 면적 특징으로 바뀌었습니다.\n    image_root = DATA_ROOT / split\n    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"\n    cache_root.mkdir(parents=True, exist_ok=True)\n    array_path = cache_root / f"{split}_maps.npy"\n    feature_path = cache_root / f"{split}_ch_features.npy"\n    metadata_path = cache_root / f"{split}_metadata.json"\n    filenames = unique_filenames(inputs)\n    expected = {\n        "image_size": IMAGE_SIZE,\n        "map_channels": list(MAP_CHANNELS),\n        "lon_limit": PROJECTION_LON_LIMIT,\n        "lat_limit": PROJECTION_LAT_LIMIT,\n        "supersample": PROJECTION_SUPERSAMPLE,\n        "limb_margin": LIMB_MARGIN,\n        "ch_ratio": ch_ratio,\n        "ch_features": list(CH_FEATURE_NAMES),\n        "geometry": [grid["geometry"] for grid in grids],\n        "filenames": filenames,\n    }\n\n    valid = False\n    if array_path.exists() and feature_path.exists() and metadata_path.exists():\n        try:\n            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n            cached = np.load(array_path, mmap_mode="r")\n            cached_features = np.load(feature_path, mmap_mode="r")\n            valid = (\n                metadata == expected\n                and cached.shape == (len(filenames), len(MAP_CHANNELS), IMAGE_SIZE, IMAGE_SIZE)\n                and cached.dtype == np.uint8\n                and cached_features.shape == (len(filenames), CH_FEATURE_DIM)\n            )\n        except (OSError, ValueError, json.JSONDecodeError):\n            valid = False\n\n    if not valid:\n        # partial 파일에 먼저 완성한 뒤 이름을 바꿔 중단된 cache의 재사용을 막습니다.\n        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")\n        feature_temp = feature_path.with_name(feature_path.name + f".partial.{os.getpid()}")\n        metadata_temp = metadata_path.with_name(metadata_path.name + f".partial.{os.getpid()}")\n        maps = np.lib.format.open_memmap(\n            array_temp, mode="w+", dtype=np.uint8,\n            shape=(len(filenames), len(MAP_CHANNELS), IMAGE_SIZE, IMAGE_SIZE),\n        )\n        features = np.zeros((len(filenames), CH_FEATURE_DIM), dtype=np.float32)\n        done = [0]\n\n        def build_row(item):\n            index, filename = item\n            projected = [\n                project(open_gray(image_root / channel / filename), grids[position])\n                for position, channel in enumerate(CHANNELS)\n            ]\n            median = float(np.median(projected[0][ON_DISK]))\n            # 논문 Fig. 4f: 193 A 맵의 동적 임계화로 코로나 홀 이진맵을 만듭니다.\n            # off-limb 격자점은 관측값이 없으므로 CH 후보에서 제외합니다.\n            binary = ((projected[0] < ch_ratio * median) & ON_DISK).astype(np.float32)\n            for position in range(len(CHANNELS)):\n                maps[index, position] = np.clip(\n                    projected[position] * ON_DISK_FLOAT, 0, 255\n                ).astype(np.uint8)\n            maps[index, len(CHANNELS)] = (binary * 255.0).astype(np.uint8)\n            features[index] = coronal_hole_features(binary)\n            done[0] += 1\n            if done[0] % 2000 == 0 or done[0] == len(filenames):\n                print(f"{split} speednet preprocess: {done[0]}/{len(filenames)}", flush=True)\n\n        with ThreadPoolExecutor(max_workers=CACHE_WORKERS) as pool:\n            list(pool.map(build_row, enumerate(filenames)))\n\n        maps.flush()\n        del maps\n        # np.save는 경로 인자에 .npy를 덧붙이므로 파일 객체로 넘깁니다.\n        with open(feature_temp, "wb") as handle:\n            np.save(handle, features)\n        metadata_temp.write_text(\n            json.dumps(expected, ensure_ascii=False) + "\\n", encoding="utf-8"\n        )\n        array_temp.replace(array_path)\n        feature_temp.replace(feature_path)\n        metadata_temp.replace(metadata_path)\n        print(f"created speednet cache: {array_path.resolve()}")\n    else:\n        print(f"reusing speednet cache: {array_path.resolve()}")\n\n    image_array = np.load(array_path, mmap_mode="r")\n    feature_array = np.load(feature_path)\n    image_index = {filename: index for index, filename in enumerate(filenames)}\n    return image_array, feature_array, image_index\n\n\n# 원반 기하 / 동적 임계값은 train split에서만 산출해 validation, test에 그대로 씁니다.\ntrain_filenames = unique_filenames(train_inputs)\nGRIDS = []\nfor channel in CHANNELS:\n    geometry = estimate_disk_geometry("train", train_filenames, channel)\n    grid = build_projection_grid(geometry)\n    grid["geometry"] = geometry\n    GRIDS.append(grid)\n\nCH_RATIO = calibrate_ch_threshold("train", train_filenames, GRIDS[0])\n\ntrain_image_array, train_ch_features, train_image_index = prepare_speednet_cache(\n    "train", train_inputs, GRIDS, CH_RATIO\n)\nval_image_array, val_ch_features, val_image_index = prepare_speednet_cache(\n    "validation", val_inputs, GRIDS, CH_RATIO\n)\ntest_image_array, test_ch_features, test_image_index = prepare_speednet_cache(\n    "test", test_inputs, GRIDS, CH_RATIO\n)\nprint(f"preprocess done at {elapsed_seconds():.0f}s (budget {TOTAL_BUDGET_SECONDS}s)")\n\n\n\n# ===== copied from code_N1.ipynb, cell 7 =====\n\ndef compute_log_statistics(image_array, sample_rows=2048):\n    # 논문 Eq.(1): 이미지별 통계가 아니라 데이터셋 전체에서 한 번 구한 고정 mu/sigma를 씁니다.\n    # (논문 값: 193 A mu=5.38 sigma=0.70 - FITS DN 기준이라 8-bit PNG인 대회 데이터에는\n    #  그대로 쓸 수 없어 같은 방식으로 train split에서 재계산합니다.)\n    rows = np.unique(\n        np.linspace(0, len(image_array) - 1, min(sample_rows, len(image_array))).astype(np.int64)\n    )\n    sample = np.log1p(np.asarray(image_array[rows][:, : len(CHANNELS)], dtype=np.float32))\n    # off-limb 격자점(값 0)은 통계에서 제외합니다.\n    sample = sample[:, :, ON_DISK]\n    mean = sample.mean(axis=(0, 2)).astype(np.float32)\n    std = np.maximum(sample.std(axis=(0, 2)), 1e-6).astype(np.float32)\n    return mean, std\n\n\nLOG_MEAN, LOG_STD = compute_log_statistics(train_image_array)\nfor position, channel in enumerate(CHANNELS):\n    print(f"AIA {channel} A : mu_log={LOG_MEAN[position]:.3f}  sigma_log={LOG_STD[position]:.3f}")\n\nWIND_FEATURE_DIM = 20 + 20 * CH_FEATURE_DIM\n\n\nclass SpeedNetDataset(Dataset):\n    def __init__(self, image_array, feature_array, image_index, inputs, indexes,\n                 log_mean, log_std, targets=None):\n        self.image_array = image_array\n        self.feature_array = feature_array\n        # CSV의 파일명을 cache row 번호로 한 번만 변환해 epoch별 문자열 탐색을 없앱니다.\n        self.image_indexes = np.asarray([\n            [image_index[filename] for filename in row]\n            for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)\n        ], dtype=np.int32)\n        # wind와 target은 약 0.x 규모로 맞춰 학습을 안정화하고, 평가 때 km/s로 복원합니다.\n        self.wind = inputs[WIND_COLUMNS].to_numpy(np.float32) / 1000.0\n        self.sample_ids = inputs.sample_id.to_numpy()\n        self.indexes = np.asarray(indexes, dtype=np.int64)\n        self.log_mean = log_mean.reshape(1, len(CHANNELS), 1, 1)\n        self.log_std = log_std.reshape(1, len(CHANNELS), 1, 1)\n        self.on_disk = ON_DISK_FLOAT.reshape(1, 1, IMAGE_SIZE, IMAGE_SIZE)\n        self.targets = targets\n\n    def __len__(self):\n        return len(self.indexes)\n\n    def __getitem__(self, item):\n        row_index = int(self.indexes[item])\n        rows = self.image_indexes[row_index]\n        raw = np.asarray(self.image_array[rows], dtype=np.float32)\n        maps = np.empty_like(raw)\n        # 논문 Eq.(1): 로그 스케일 후 고정 mu/sigma 표준화. CH 이진맵은 0/1 그대로 둡니다.\n        # off-limb 격자점은 표준화 후 0(분포의 평균)으로 눌러 극단값이 들어가지 않게 합니다.\n        maps[:, : len(CHANNELS)] = (\n            (np.log1p(raw[:, : len(CHANNELS)]) - self.log_mean) / self.log_std\n        ) * self.on_disk\n        maps[:, len(CHANNELS)] = raw[:, len(CHANNELS)] / 255.0\n        # wind 20개 + 시점별 CH 면적 특징 20x4개\n        wind = np.concatenate(\n            [self.wind[row_index], self.feature_array[rows].ravel()]\n        ).astype(np.float32)\n        # 반환 shape: images=(20,3,64,64), wind=(100,), target=(12,)\n        result = {\n            "images": torch.from_numpy(maps),\n            "wind": torch.from_numpy(wind),\n            "sample_id": self.sample_ids[row_index],\n        }\n        if self.targets is not None:\n            result["target"] = torch.from_numpy(\n                np.asarray(self.targets[row_index], dtype=np.float32) / 1000.0\n            )\n        return result\n\n\ndef seed_worker(worker_id):\n    worker_seed = (SEED + worker_id) % (2 ** 32)\n    random.seed(worker_seed)\n    np.random.seed(worker_seed)\n\n\ndef make_loader(dataset, shuffle):\n    options = dict(\n        dataset=dataset,\n        batch_size=BATCH_SIZE,\n        shuffle=shuffle,\n        num_workers=NUM_WORKERS,\n        pin_memory=PIN_MEMORY,\n        drop_last=False,\n        worker_init_fn=seed_worker,\n        generator=torch.Generator().manual_seed(SEED),\n    )\n    if NUM_WORKERS > 0:\n        options.update(persistent_workers=True, prefetch_factor=4)\n    return DataLoader(**options)\n\n\ntrain_dataset = SpeedNetDataset(\n    train_image_array, train_ch_features, train_image_index, train_inputs, train_index,\n    LOG_MEAN, LOG_STD, train_targets,\n)\nval_dataset = SpeedNetDataset(\n    val_image_array, val_ch_features, val_image_index, val_inputs, val_index,\n    LOG_MEAN, LOG_STD, val_targets,\n)\ntrain_loader = make_loader(train_dataset, shuffle=True)\nval_loader = make_loader(val_dataset, shuffle=False)\n\nbatch = next(iter(train_loader))\nassert batch["images"].shape[1:] == (20, len(MAP_CHANNELS), IMAGE_SIZE, IMAGE_SIZE)\nassert batch["wind"].shape[1:] == (WIND_FEATURE_DIM,)\nassert batch["target"].shape[1:] == (12,)\nprint(batch["images"].shape, batch["wind"].shape, batch["target"].shape)\n\n\n\n# ===== copied from code_N1.ipynb, cell 10 =====\n\nclass SpeedNetConvBlock(nn.Module):\n    # 논문 Fig. 7의 SpeedNet Conv. Block: Convolution layer -> ReLU -> MaxPooling\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)\n        self.pool = nn.MaxPool2d(2)\n\n    def forward(self, x):\n        return self.pool(F.relu(self.conv(x), inplace=True))\n\n\nclass SpeedNet(nn.Module):\n    def __init__(self, lstm_units=LSTM_UNITS, ffnn_units=FFNN_UNITS):\n        super().__init__()\n        blocks = []\n        in_channels = len(MAP_CHANNELS)\n        for out_channels in CONV_CHANNELS:\n            blocks.append(SpeedNetConvBlock(in_channels, out_channels))\n            in_channels = out_channels\n        self.conv_blocks = nn.Sequential(*blocks)\n        spatial = IMAGE_SIZE // (2 ** len(CONV_CHANNELS))\n        self.feature_dim = CONV_CHANNELS[-1] * spatial * spatial\n        self.lstm = nn.LSTM(\n            input_size=self.feature_dim,\n            hidden_size=lstm_units,\n            batch_first=True,\n        )\n        self.wind_encoder = nn.Sequential(\n            nn.Linear(WIND_FEATURE_DIM, 128), nn.SELU(inplace=True),\n            nn.Linear(128, 64), nn.SELU(inplace=True),\n        )\n        # 논문 Fig. 7의 FFNN. 출력 유닛만 1개 -> 12개(6~72시간)로 바꿨습니다.\n        self.head = nn.Sequential(\n            nn.Linear(lstm_units + 64, ffnn_units), nn.ReLU(inplace=True),\n            nn.Linear(ffnn_units, 12),\n        )\n\n    def forward(self, images, wind):\n        batch_size, steps = images.shape[0], images.shape[1]\n        # 가중치를 공유하는 Conv. Block을 20시점에 각각 적용합니다.\n        image_features = self.conv_blocks(images.flatten(0, 1))\n        image_features = image_features.reshape(batch_size, steps, -1)\n        _, (hidden, _) = self.lstm(image_features)\n        image_features = F.relu(hidden[-1])\n        wind_features = self.wind_encoder(wind)\n        return self.head(torch.cat([image_features, wind_features], dim=1))\n\n\nmodel = SpeedNet().to(DEVICE)\ntrainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)\nprint(model)\nprint("trainable parameters:", f"{trainable_parameters:,}")\n\n\n\n# ===== copied from code_N1.ipynb, cell 12 =====\n\n# 매 학습 실행마다 새 모델을 만들며 pretrained/이전 실행 weight는 불러오지 않습니다.\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\nmodel = SpeedNet().to(DEVICE)\n# 논문 Table 4: Adam, lr=0.0001 고정 (learning rate scheduler 없음)\noptimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)\n# AMP는 GPU 메모리 사용량을 줄이고 연산을 가속합니다.\nscaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)\ncheckpoint_path = OUTPUT_DIR / "best_model.pth"\nif checkpoint_path.exists():\n    checkpoint_path.unlink()\nprint("initialized model from scratch; removed any previous checkpoint")\n\nbest_val_rmse = float("inf")\nepochs_without_improvement = 0\nhistory = []\n\ntraining_started = time.perf_counter()\ntrain_budget_seconds = max(60.0, remaining_seconds() - INFERENCE_RESERVE_SECONDS)\nprint(f"training budget: {train_budget_seconds:.0f}s")\n\n\ndef run_epoch(loader, training):\n    model.train(training)\n    squared_error_sum = 0.0\n    value_count = 0\n    for batch in loader:\n        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)\n        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)\n        if training:\n            optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training):\n            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n                prediction = model(images, wind)\n                # 논문 Table 4의 loss function: Mean Squared Error\n                loss = F.mse_loss(prediction, target)\n            if training:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n        # 사용자에게 익숙한 km/s 단위로 되돌려 전체 원소 기준 RMSE를 누적합니다.\n        error_km_s = (prediction.detach().float() - target) * 1000.0\n        squared_error_sum += float(torch.sum(error_km_s ** 2).cpu())\n        value_count += error_km_s.numel()\n    return math.sqrt(squared_error_sum / value_count)\n\n\nfor epoch in range(1, EPOCHS + 1):\n    started = time.perf_counter()\n    train_rmse = run_epoch(train_loader, training=True)\n    with torch.no_grad():\n        val_rmse = run_epoch(val_loader, training=False)\n    elapsed = time.perf_counter() - started\n    history.append({\n        "epoch": epoch,\n        "train_rmse_km_s": train_rmse,\n        "val_rmse_km_s": val_rmse,\n        "learning_rate": LEARNING_RATE,\n        "seconds": elapsed,\n    })\n    print(\n        f"epoch={epoch:03d} train_rmse={train_rmse:.3f} "\n        f"val_rmse={val_rmse:.3f} lr={LEARNING_RATE:.2e} "\n        f"seconds={elapsed:.1f} total={elapsed_seconds():.0f}s"\n    )\n    # 논문 Table 4의 model callback: validation loss가 가장 낮은 epoch만 보관합니다.\n    if val_rmse < best_val_rmse:\n        best_val_rmse = val_rmse\n        epochs_without_improvement = 0\n        torch.save(\n            {\n                "model_state_dict": model.state_dict(),\n                "epoch": epoch,\n                "val_rmse_km_s": val_rmse,\n                "channels": CHANNELS,\n                "map_channels": MAP_CHANNELS,\n                "image_size": IMAGE_SIZE,\n                "log_mean": LOG_MEAN,\n                "log_std": LOG_STD,\n                "ch_ratio": CH_RATIO,\n                "geometry": [grid["geometry"] for grid in GRIDS],\n                "hyperparameters": {\n                    "lstm_units": LSTM_UNITS,\n                    "ffnn_units": FFNN_UNITS,\n                    "conv_channels": list(CONV_CHANNELS),\n                    "learning_rate": LEARNING_RATE,\n                    "batch_size": BATCH_SIZE,\n                },\n                "initialization": "random_from_scratch",\n            },\n            checkpoint_path,\n        )\n    else:\n        epochs_without_improvement += 1\n        # 논문 Table 3: early stopping patience = 20\n        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:\n            print("early stopping")\n            break\n    # 20분 예산: 다음 epoch을 돌릴 시간이 없으면 여기서 멈추고 추론으로 넘어갑니다.\n    spent = time.perf_counter() - training_started\n    if spent + elapsed > train_budget_seconds:\n        print(f"time budget reached ({spent:.0f}s); stopping training")\n        break\n\nhistory_frame = pd.DataFrame(history)\nhistory_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)\nhistory_frame.plot(x="epoch", y=["train_rmse_km_s", "val_rmse_km_s"], grid=True)\nplt.ylabel("RMSE (km/s)")\nplt.tight_layout()\nplt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=140)\nplt.show()\n\ncheckpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)\nmodel.load_state_dict(checkpoint["model_state_dict"])\nprint("loaded best epoch:", checkpoint["epoch"],\n      "val_rmse:", round(checkpoint["val_rmse_km_s"], 3))\n\n\n\n# ===== copied from code_N1.ipynb, cell 14 =====\n\n@torch.no_grad()\ndef predict(loader):\n    model.eval()\n    predictions = []\n    sample_ids = []\n    for batch in loader:\n        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)\n        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)\n        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):\n            output = model(images, wind)\n        predictions.append(output.float().cpu().numpy() * 1000.0)\n        sample_ids.extend(batch["sample_id"])\n    return np.concatenate(predictions), sample_ids\n\n\nvalidation_prediction, validation_ids = predict(val_loader)\nvalidation_target = np.asarray(val_targets[val_index], dtype=np.float64)\nassert validation_ids == val_inputs.iloc[val_index].sample_id.tolist()\n\n# persistence baseline: 마지막 관측 wind가 72시간 동안 유지된다고 가정\nvalidation_persistence = np.repeat(\n    val_inputs.iloc[val_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64), 12, axis=1\n)\n\n\ndef metrics_by_horizon(y_true, y_pred):\n    rows = []\n    for index in range(12):\n        actual = y_true[:, index]\n        predicted = y_pred[:, index]\n        error = predicted - actual\n        denominator = np.std(actual) * np.std(predicted)\n        correlation = (\n            float(np.corrcoef(actual, predicted)[0, 1]) if denominator > 0 else np.nan\n        )\n        rows.append({\n            "horizon_hours": (index + 1) * 6,\n            "rmse_km_s": float(np.sqrt(np.mean(error ** 2))),\n            "mae_km_s": float(np.mean(np.abs(error))),\n            "correlation": correlation,\n        })\n    return pd.DataFrame(rows)\n\n\nvalidation_metrics = metrics_by_horizon(validation_target, validation_prediction)\npersistence_metrics = metrics_by_horizon(validation_target, validation_persistence)\nvalidation_metrics["persistence_rmse_km_s"] = persistence_metrics.rmse_km_s\nvalidation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)\nprint(\n    "overall validation RMSE:",\n    float(np.sqrt(np.mean((validation_prediction - validation_target) ** 2))),\n)\nprint(\n    "persistence  validation RMSE:",\n    float(np.sqrt(np.mean((validation_persistence - validation_target) ** 2))),\n)\nvalidation_metrics\n\n\n\n# ===== copied from code_N1.ipynb, cell 16 =====\n\n# test 추론 전에 학습용 객체와 CUDA cache를 정리해 불필요한 메모리를 반환합니다.\ndel train_loader, val_loader, train_dataset, val_dataset\ngc.collect()\nif DEVICE.type == "cuda":\n    torch.cuda.empty_cache()\n\ntest_dataset = SpeedNetDataset(\n    test_image_array, test_ch_features, test_image_index, test_inputs, test_index,\n    LOG_MEAN, LOG_STD, targets=None,\n)\ntest_loader = make_loader(test_dataset, shuffle=False)\ntest_prediction, predicted_ids = predict(test_loader)\nexpected_ids = test_inputs.iloc[test_index].sample_id.tolist()\nassert predicted_ids == expected_ids\nassert test_prediction.shape == (len(test_index), 12)\nassert np.isfinite(test_prediction).all()\n\n# 제출 형식은 sample_id 다음에 target_00~target_11이 오는 13개 column입니다.\nsubmission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)\nsubmission.insert(0, "sample_id", predicted_ids)\nsubmission.to_csv(OUTPUT_DIR / "submission.csv", index=False)\nsubmission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)\ntorch.save(checkpoint, SUBMISSION_DIR / "model.pth")\n\nassert submission.columns.tolist() == ["sample_id"] + TARGET_COLUMNS\nassert submission.sample_id.is_unique\nassert len(submission) == len(test_inputs)\n\nprint("saved:", (SUBMISSION_DIR / "submission.csv").resolve())\nprint("saved:", (SUBMISSION_DIR / "model.pth").resolve())\nprint("shape:", submission.shape)\nprint(f"total elapsed: {elapsed_seconds():.0f}s / budget {TOTAL_BUDGET_SECONDS}s")\n\ndel test_dataset, test_loader\ngc.collect()\nif DEVICE.type == "cuda":\n    torch.cuda.empty_cache()\n\nsubmission.head()\n\n\n', N1)

## 4. 사전 고정한 P3 + P9 + N1 동일 가중치 평균

In [ ]:
# ========================== P12 equal-weight ensemble ==========================
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

P3_VAL = np.asarray(P3["validation_prediction"], dtype=np.float32)
P3_TEST = np.asarray(P3["test_prediction"], dtype=np.float32)
P9_VAL = np.asarray(P9["val_prediction"], dtype=np.float32)
P9_TEST = np.asarray(P9["test_prediction"], dtype=np.float32)
N1_VAL = np.asarray(N1["validation_prediction"], dtype=np.float32)
N1_TEST = np.asarray(N1["test_prediction"], dtype=np.float32)
VAL_TARGETS = np.asarray(P3["val_targets"], dtype=np.float32)
SAMPLE_IDS = P3["test_inputs"].sample_id.tolist()

assert P3_TEST.shape == P9_TEST.shape == N1_TEST.shape == (len(SAMPLE_IDS), 12)
assert P3_VAL.shape == P9_VAL.shape == N1_VAL.shape == VAL_TARGETS.shape
assert P3["test_inputs"].sample_id.tolist() == P9["test_inputs"].sample_id.tolist()
assert P3["test_inputs"].sample_id.tolist() == N1["test_inputs"].sample_id.tolist()

# Fixed before inspecting this run: no fitted blend weights, no validation member selection.
MEMBER_NAMES = ("P3", "P9_F3", "N1")
WEIGHTS = np.array((1 / 3, 1 / 3, 1 / 3), dtype=np.float32)
VAL_STACK = np.stack((P3_VAL, P9_VAL, N1_VAL))
TEST_STACK = np.stack((P3_TEST, P9_TEST, N1_TEST))
FINAL_VAL = np.tensordot(WEIGHTS, VAL_STACK, axes=(0, 0))
FINAL_TEST = np.tensordot(WEIGHTS, TEST_STACK, axes=(0, 0))

def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon

for name, pred in zip(MEMBER_NAMES, VAL_STACK):
    print(f"{name:7s} validation RMSE: {official_rmse(VAL_TARGETS, pred)[0]:.3f}")
print(f"P12    validation RMSE: {official_rmse(VAL_TARGETS, FINAL_VAL)[0]:.3f}")
pairwise = {
    f"{MEMBER_NAMES[i]}__{MEMBER_NAMES[j]}": float(np.sqrt(np.mean((TEST_STACK[i] - TEST_STACK[j]) ** 2)))
    for i in range(3) for j in range(i + 1, 3)
}
print("test prediction disagreement (RMS km/s):", {k: round(v, 2) for k, v in pairwise.items()})

SUBMISSION_DIR = Path("submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]
submission = pd.DataFrame(FINAL_TEST, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", SAMPLE_IDS)
assert submission.shape == (len(SAMPLE_IDS), 13)
assert submission.sample_id.is_unique and np.isfinite(FINAL_TEST).all()
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

def cpu_state(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}

# State dicts make all three trained members inspectable/reloadable from one artifact.
torch.save({
    "code_version": "P12",
    "ensemble": "fixed equal-weight mean; no fitted blend parameters",
    "weights": WEIGHTS,
    "members": {
        "P3": {"state_dict": cpu_state(P3["model"]),
               "best_epoch": int(P3["checkpoint"]["epoch"]),
               "recipe": "P3 original: AdamW + ReduceLROnPlateau + validation early stopping"},
        "P9_F3": {"state_dict": cpu_state(P9["model"]),
                  "epochs": int(P9["P9_FINAL_EPOCHS"]),
                  "settings": P9["P9_SETTINGS"],
                  "recipe": "P9 adopted F3, fixed one epoch"},
        "N1": {"state_dict": cpu_state(N1["model"]),
               "best_epoch": int(N1["checkpoint"]["epoch"]),
               "recipe": "N1 SpeedNet original: Adam + validation best checkpoint"},
    },
    "test_sample_ids": SAMPLE_IDS,
}, SUBMISSION_DIR / "model.pth")
print(f"saved {SUBMISSION_DIR / 'submission.csv'}: {submission.shape}")
print(f"saved {SUBMISSION_DIR / 'model.pth'}: three member state dicts")


## 5. 저장 상태만으로 재추론 검증

In [ ]:
# ================================ Reproducibility check ================================
# Use the in-notebook preprocessing and the three saved states to regenerate test predictions.
blob = torch.load(SUBMISSION_DIR / "model.pth", map_location="cpu", weights_only=False)
assert tuple(blob["members"]) == MEMBER_NAMES
assert np.allclose(blob["weights"], WEIGHTS)

def reload_predict_p3():
    ns = P3
    model = ns["build_model"]()
    model.load_state_dict(blob["members"]["P3"]["state_dict"])
    model.eval()
    ns["model"] = model
    dataset = ns["SolarWindDataset"](ns["test_image_array"], ns["test_image_index"], ns["test_inputs"],
                                      ns["test_wind"], ns["test_wind_valid"], ns["test_ch"], targets=None)
    loader = ns["make_loader"](dataset, shuffle=False)
    prediction, ids = ns["predict"](loader)
    assert ids == SAMPLE_IDS
    del model, dataset, loader
    return prediction

def reload_predict_p9():
    ns = P9
    ns["configure"](**ns["P9_SETTINGS"])
    model = ns["build_model"](ns["FINAL_STATS"])
    model.load_state_dict(blob["members"]["P9_F3"]["state_dict"])
    ns["model"] = model
    prediction, ids = ns["p9_predict"](
        ns["test_inputs"], ns["test_index_matrix"], ns["test_wind"], ns["test_wind_valid"], ns["test_grid"],
    )
    assert ids == SAMPLE_IDS
    return prediction

def reload_predict_n1():
    ns = N1
    model = ns["SpeedNet"]().to(ns["DEVICE"])
    model.load_state_dict(blob["members"]["N1"]["state_dict"])
    ns["model"] = model
    dataset = ns["SpeedNetDataset"](ns["test_image_array"], ns["test_ch_features"],
                                     ns["test_image_index"], ns["test_inputs"], ns["test_index"],
                                     ns["LOG_MEAN"], ns["LOG_STD"], targets=None)
    loader = ns["make_loader"](dataset, shuffle=False)
    prediction, ids = ns["predict"](loader)
    assert ids == SAMPLE_IDS
    del dataset, loader
    return prediction

reloaded = np.tensordot(WEIGHTS, np.stack((reload_predict_p3(), reload_predict_p9(),
                                            reload_predict_n1())), axes=(0, 0))
saved = pd.read_csv(SUBMISSION_DIR / "submission.csv")[TARGET_COLUMNS].to_numpy(np.float32)
gap = float(np.sqrt(np.mean((reloaded - saved) ** 2)))
print(f"reproduction RMS error: {gap:.6f}")
assert gap < 0.01, "saved model states do not reproduce submission.csv"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. 제출 점검

In [ ]:
# ================================ Submission check ================================
import shutil

EXPECTED_TEST_ROWS = 3868
assert submission.shape == (EXPECTED_TEST_ROWS, 13), submission.shape
assert submission.columns.tolist() == ["sample_id"] + TARGET_COLUMNS
assert np.isfinite(submission[TARGET_COLUMNS].to_numpy()).all()
assert (SUBMISSION_DIR / "model.pth").is_file()

# Save this notebook as code_p12.ipynb before running this cell; it is then copied to the required name.
notebook_here = Path("code_p12.ipynb")
if notebook_here.exists():
    shutil.copyfile(notebook_here, SUBMISSION_DIR / "code.ipynb")
    print("code_p12.ipynb -> submission/code.ipynb")
else:
    print("Save this notebook as code_p12.ipynb, then copy it to submission/code.ipynb.")
for path in (SUBMISSION_DIR / "code.ipynb", SUBMISSION_DIR / "model.pth", SUBMISSION_DIR / "submission.csv"):
    print(path, "OK" if path.exists() else "MISSING")
